# Medical Health Insurance — Portfolio Analysis

One extract: 100,000 members, one year each, 54 fields. This notebook runs top to bottom —
document the schema, inspect the raw values, clean, explore, chart, model.

| Section | Question |
|:---|:---|
| 1. The fields I have been given | What does each column mean, and who uses it? |
| 2. Looking at the raw extract | What condition is this data in? |
| 3. Cleaning | Which records cannot describe a real person? |
| 4. Exploring the book | What drives cost, and what can this data not answer? |
| 5. The dashboard picture | The headline charts |
| 6. Predicting cost and risk | What will a member cost, and who becomes expensive? |

Two constraints established below and enforced throughout. **`Annual Premium` is not an
underwritten price** (4.1), so no pricing-adequacy or profitability claim is available from this
book. And **the analysis frame is not the modelling frame** — section 6 builds a separate one.

---

# 1. The Fields I Have Been Given

Before I open anything I want to write down what each column is supposed to mean, who in the
business would use it, and which ones are going to cause trouble later. This section documents
and profiles. It changes nothing.

## Setting Up

Two packages to start with — `pathlib` to find the file and `pandas` to read it. Everything
else arrives at the section that first needs it.

In [ ]:
# Importing what I need to open the file
from pathlib import Path

import pandas as pd

In [ ]:
# Widening the display, since 54 columns will not fit at the default settings
pd.set_option('display.max_columns', 70)
pd.set_option('display.max_rows', 70)
pd.set_option('display.width', 1000)

I am resolving the project root by looking for the data folder rather than assuming a working
directory, so this runs whether the Jupyter server was started at the root or somewhere below
it.

In [ ]:
# Finding the project root by looking for the data folder, rather than assuming where I started
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'data' / 'medical_insurance.csv').exists())

RAW_PATH = ROOT / 'data' / 'medical_insurance.csv'
PROCESSED_DIR = ROOT / 'data' / 'processed'
FIGS = ROOT / 'reports' / 'figures'
MODELS = ROOT / 'updated_models'    # kept separate from models/, which holds earlier runs

for d in (PROCESSED_DIR, FIGS, MODELS):
    d.mkdir(parents=True, exist_ok=True)

ROOT

In [ ]:
# Reading the raw extract
df = pd.read_csv(RAW_PATH)
df.shape

100,000 members and 54 fields.

## Who Uses What

The business has several audiences for this data and they want different things from it. I am
using these codes throughout, so that every conclusion below can be addressed to somebody in
particular rather than left hanging.

| Code | Stakeholder | What they want from this data |
|:---|:---|:---|
| UW | Underwriting and risk selection | Assess member risk, review eligibility |
| ACT | Actuarial and pricing | Estimate claims, frequency, severity, adequacy |
| CLM | Claims operations | Process and control claims |
| CM | Care management | Decide which members to enrol in a programme |
| FIN | Finance | Revenue, cost, profitability |
| RE | Reinsurance and capital | Severe and catastrophic exposure |

## The Column Dictionary

### Identity and demographics

| Column | Meaning | Insurance use | Users |
|:---|:---|:---|:---|
| `person_id` | Unique member identifier | Joins records; never a risk feature | UW, ACT, CLM |
| `age` | Member age in years | Age-related morbidity and care planning | UW, ACT |
| `sex` | Recorded sex category | Population risk; review for fairness | ACT, UW |
| `region` | Broad geographic area | Regional cost, access and market analysis | ACT |
| `urban_rural` | Settlement classification | Healthcare access and utilisation | ACT |
| `income` | Annual income estimate | Affordability and segmentation | ACT, FIN |
| `education` | Highest education level | Health literacy and segmentation | ACT, UW |
| `marital_status` | Marital status | Household and family cover | ACT |
| `employment_status` | Employment category | Segments individual and employment-linked pools | UW, ACT |
| `household_size` | People in household | Family exposure and product design | ACT |
| `dependents` | Number of dependents | Family pricing and coverage | ACT, FIN |

### Lifestyle and clinical measurements

| Column | Meaning | Insurance use | Users |
|:---|:---|:---|:---|
| `bmi` | Body mass index | Obesity-related risk, wellness targeting | UW, ACT |
| `smoker` | Never, former or current | Respiratory, cardiovascular and cancer risk | UW, ACT |
| `alcohol_freq` | Reported drinking frequency | Lifestyle risk; **missing means unknown** | UW |
| `systolic_bp` | Upper blood-pressure reading | Hypertension and cardiovascular risk | UW |
| `diastolic_bp` | Lower blood-pressure reading | Blood-pressure control | UW |
| `ldl` | LDL cholesterol | Cardiovascular risk | UW, ACT |
| `hba1c` | Average blood glucose over 2-3 months | Diabetes risk and care management | UW, ACT |
| `medication_count` | Active medications | Chronic burden and polypharmacy | UW, ACT |

### Utilisation and chronic conditions

| Column | Meaning | Insurance use | Users |
|:---|:---|:---|:---|
| `visits_last_year` | Outpatient visits in the prior 12 months | Care demand | CLM, ACT |
| `hospitalizations_last_3yrs` | Admissions in the prior three years | Prior severe events | CLM, ACT |
| `days_hospitalized_last_3yrs` | Inpatient days over three years | Length of stay and severity | CLM, ACT, RE |
| `chronic_count` | Number of chronic conditions | Comorbidity segmentation | UW, ACT |
| `hypertension`, `diabetes`, `asthma`, `copd`, `cardiovascular_disease`, `cancer_history`, `kidney_disease`, `liver_disease`, `arthritis`, `mental_health` | Ten condition flags, 0 or 1 | Morbidity mix and cost risk | UW, ACT, CM, RE |

### Procedure mix

| Column | Meaning | Insurance use | Users |
|:---|:---|:---|:---|
| `proc_imaging_count` | X-ray, CT, MRI | Diagnostic intensity | CLM, ACT |
| `proc_surgery_count` | Surgeries | Severity and reinsurance exposure | CLM, RE |
| `proc_physio_count` | Physiotherapy sessions | Rehabilitation utilisation | CLM |
| `proc_consult_count` | Specialist consultations | Referral patterns | CLM, ACT |
| `proc_lab_count` | Laboratory tests | Diagnostic and monitoring intensity | CLM, ACT |

### Product

| Column | Meaning | Insurance use | Users |
|:---|:---|:---|:---|
| `plan_type` | HMO, PPO, EPO or POS | Network and utilisation analysis | ACT |
| `network_tier` | Bronze through Platinum | Benefit richness | ACT, FIN |
| `deductible` | Paid by the member before cover contributes | Cost sharing and plan design | ACT |
| `copay` | Fixed fee per visit | Utilisation steering | ACT |
| `policy_term_years` | Policy tenure | Retention and maturity | ACT, FIN |
| `policy_changes_last_2yrs` | Changes in the prior two years | Engagement and lapse risk | UW |
| `provider_quality` | Provider rating, roughly 1.5 to 5.0 | Network contracting | ACT |
| `risk_score` | Composite member risk, 0 to 1 | Underwriting triage; **check for leakage** | UW, ACT |

### Financial and target fields

| Column | Meaning | Insurance use | Users |
|:---|:---|:---|:---|
| `annual_medical_cost` | Total medical spending for the year | The cost target | ACT, FIN, RE |
| `annual_premium` | Annual premium charged | Revenue; **check how it was set** | ACT, FIN |
| `monthly_premium` | Monthly premium | Billing; likely derived from annual | FIN |
| `claims_count` | Claims submitted | Frequency | ACT, CLM |
| `avg_claim_amount` | Average paid per claim | Severity | ACT, CLM, RE |
| `total_claims_paid` | Total paid on claims | Loss ratio; **likely derived from cost** | ACT, FIN |
| `is_high_risk` | Existing high-risk flag | Triage; candidate target | UW |
| `had_major_procedure` | Major procedure flag | Severe-event monitoring | CLM, RE |

## Profiling Every Column

Before I read a single value I want the shape of each field — what type it holds, how much of it
is missing, and how many distinct values it takes. A field with two distinct values is a flag; a
field with 100,000 is an identifier; a field with nothing missing is one less thing to worry
about.

In [ ]:
# Profiling every column: type, completeness, and how many distinct values it holds
profile = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing': df.isna().sum(),
    'missing %': (df.isna().mean() * 100).round(2),
    'distinct': df.nunique(dropna=False),
})
profile

Only one field has anything missing, and the identifier is unique on every row. Both of those
are worth confirming rather than assuming, so I look at them properly in the next section.

## Checks I Can Run Without Changing Anything

Several fields in the dictionary look as though they might be calculated from other fields. If
they are, that matters enormously later — a column computed from the answer is not a predictor,
it is the answer wearing a different name.

I am testing those suspicions now, while nothing has been touched, so the results describe the
source file rather than anything I did to it.

In [ ]:
# Testing the identities the dictionary hints at, before anything is modified
chronic_cols = ['hypertension', 'diabetes', 'asthma', 'copd', 'cardiovascular_disease',
                'cancer_history', 'kidney_disease', 'liver_disease', 'arthritis', 'mental_health']

pd.Series({
    'duplicate person_id': int(df['person_id'].duplicated().sum()),
    'chronic_count != sum of the ten flags': int((df['chronic_count'] != df[chronic_cols].sum(axis=1)).sum()),
    'worst gap: annual_premium vs 12 x monthly': float((df['annual_premium'] - 12 * df['monthly_premium']).abs().max()),
    'worst gap: total_claims_paid vs count x average': float((df['total_claims_paid'] - df['claims_count'] * df['avg_claim_amount']).abs().max()),
    'members recorded at age 0': int((df['age'] == 0).sum()),
    'alcohol_freq missing': int(df['alcohol_freq'].isna().sum()),
}, name='value')

Four results, and each one sets up work further down.

`person_id` is unique and `chronic_count` reconciles exactly against its ten flags, so the file
is internally consistent where it claims to be.

**`monthly_premium` is `annual_premium / 12` to the cent, and `total_claims_paid` is
`claims_count × avg_claim_amount` to the cent.** Neither field carries anything the other two do
not. That is the first sign of a pattern I chase down properly in section 4.

**165 members are recorded at age 0**, and **30,083 have no alcohol frequency recorded** — 30% of
the book in a single field. Both need investigating before I decide anything, which is section 2.

## What I Will and Will Not Model

Recorded now, before seeing which fields score well.

| Objective | Target | Fields needing a leakage review |
|:---|:---|:---|
| What will a member cost? | `annual_medical_cost` | Premium fields, claims-paid fields, `risk_score` |
| Who becomes expensive? | Top decile of `annual_medical_cost` | The same, plus anything recorded in the same year |
| What should we charge? | No model | Premium looks computed from cost — tested in 4.1 |

`person_id` is never a predictor and a target is never an input. Feature selection also depends on
**when** the prediction is made: a field present in the file is not necessarily available when
somebody needs the answer.

---

# 2. Looking at the Raw Extract

First read of the actual values. Nothing is corrected here — the repairs are section 3.

Two things happen beyond looking. The columns get readable names, because `proc_physio_count` and
`hospitalizations_last_3yrs` invite mistakes. And where one field can be checked against another,
that comparison is built now for cleaning to use.

Neither changes a member's data.

In [ ]:
# Adding numpy for the comparisons I am about to build
import numpy as np

## Giving the Columns Readable Names

Not cosmetic. I will be typing and reading these constantly, and `proc_physio_count` invites
mistakes in a way `Physiotherapy Procedures Count` does not.

I am asserting the map has exactly 54 entries, so that if a column is ever added to the source
file this cell fails loudly instead of silently leaving one behind.

In [ ]:
# Mapping every database column name to something readable
COLUMN_RENAME_MAP = {
    'person_id': 'Id',
    'age': 'Age',
    'sex': 'Sex',
    'region': 'Region',
    'urban_rural': 'Urban / Rural',
    'income': 'Income',
    'education': 'Education (Qualification)',
    'marital_status': 'Marital Status',
    'employment_status': 'Employment Status',
    'household_size': 'Household Size',
    'dependents': 'Dependents',
    'bmi': 'BMI',
    'smoker': 'Smoker Status',
    'alcohol_freq': 'Alcohol Frequency',
    'visits_last_year': 'Visits in Last Year',
    'hospitalizations_last_3yrs': 'Hospitalizations in Last 3 Years',
    'days_hospitalized_last_3yrs': 'Days Hospitalized in Last 3 Years',
    'medication_count': 'Medication Count',
    'systolic_bp': 'Systolic Blood Pressure',
    'diastolic_bp': 'Diastolic Blood Pressure',
    'ldl': 'LDL Cholesterol',
    'hba1c': 'HbA1c Level',
    'plan_type': 'Plan Type',
    'network_tier': 'Network Tier',
    'deductible': 'Deductible',
    'copay': 'Copay',
    'policy_term_years': 'Policy Term (Years)',
    'policy_changes_last_2yrs': 'Policy Changes in Last 2 Years',
    'provider_quality': 'Provider Quality Rating',
    'risk_score': 'Risk Score',
    'annual_medical_cost': 'Annual Medical Cost',
    'annual_premium': 'Annual Premium',
    'monthly_premium': 'Monthly Premium',
    'claims_count': 'Claims Count',
    'avg_claim_amount': 'Average Claim Amount',
    'total_claims_paid': 'Total Claims Paid',
    'chronic_count': 'Chronic Conditions Count',
    'hypertension': 'Hypertension',
    'diabetes': 'Diabetes',
    'asthma': 'Asthma',
    'copd': 'COPD',
    'cardiovascular_disease': 'Cardiovascular Disease',
    'cancer_history': 'Cancer History',
    'kidney_disease': 'Kidney Disease',
    'liver_disease': 'Liver Disease',
    'arthritis': 'Arthritis',
    'mental_health': 'Mental Health Condition',
    'proc_imaging_count': 'Imaging Procedures Count',
    'proc_surgery_count': 'Surgical Procedures Count',
    'proc_physio_count': 'Physiotherapy Procedures Count',
    'proc_consult_count': 'Consultation Procedures Count',
    'proc_lab_count': 'Lab Procedures Count',
    'is_high_risk': 'Is High Risk',
    'had_major_procedure': 'Had Major Procedure',
}

assert len(COLUMN_RENAME_MAP) == 54, f'Expected 54 columns, got {len(COLUMN_RENAME_MAP)}'

df = df.rename(columns=COLUMN_RENAME_MAP)
df.head(3)

## Is There One Row Per Person?

This settles the **grain**, which is the question of what a single row represents. It matters
because everything downstream assumes it: if a member could appear twice I would be
double-counting their cost, and every average in the analysis would be wrong.

In [ ]:
# Checking the grain - one row per member, or can a member appear twice?
pd.Series({
    'rows': len(df),
    'unique member ids': df['Id'].nunique(),
    'fully duplicated rows': int(df.duplicated().sum()),
})

100,000 unique member IDs across 100,000 rows, and no duplicated rows. One row is one member for
one year, and I can treat each row as one member-year of exposure.

## What Is Missing, and What Type Is Everything

Two questions I always ask of a new dataset: is anything missing, and is every field stored as
the right kind of value. The second one catches a common and quiet problem — a number stored as
text, money with a currency symbol in it — which silently breaks any arithmetic done on it.

In [ ]:
# Counting missing values, showing only the columns that have any
missing = df.isnull().sum()
missing[missing > 0]

Only one column has anything missing, and it has a lot: **`Alcohol Frequency` is blank for 30,083
members**, which is 30% of the book. Every other one of the 54 fields is complete.

A single column carrying all of the missing data is unusual and it points somewhere specific. If
records had been damaged in transfer I would expect gaps scattered across many columns. Missing
data concentrated in exactly one field usually means that field was optional at the point of
collection — which is a question about the survey, not about the data pipeline. I chase it down
further below.

In [ ]:
# Checking which fields are stored as text, in case a number is hiding in one
df.select_dtypes(include='str').columns.tolist()

All nine text fields are genuine categories — sex, region, education and so on. No numbers are
hiding in text form, so I can do arithmetic on the numeric fields without converting anything
first.

The binary condition flags being stored as whole numbers rather than true and false is fine, and
actually convenient: storing them as 0 and 1 means I can average the column to get a prevalence
rate directly.

In [ ]:
# Looking at every distinct value in each text field, where messy data usually shows itself
pd.Series({col: sorted(df[col].dropna().unique()) for col in df.select_dtypes(include='str').columns})

These are clean. No duplicated categories from inconsistent spelling, no stray whitespace, no
surprise values.

Two things I want to change, though, both about clarity rather than correctness. The education
values use abbreviations — `HS`, `No HS`, `Some College` — that will read badly on a chart axis.
And the plan types are four-letter acronyms that mean nothing to a reader who does not already
know them.

## Making the Labels Readable

I am expanding the education values in place. For plan type I am **adding** a new column with the
full name rather than replacing the original, so the short code stays available for anything that
needs it while charts can use the readable version.

In [ ]:
# Spelling out the education values
EDUCATION_MAP = {
    'No HS': 'Primary',
    'HS': 'High School',
    'Some College': 'College',
    'Bachelors': 'Bachelors',
    'Masters': 'Masters',
    'Doctorate': 'Doctorate',
}

df['Education (Qualification)'] = df['Education (Qualification)'].map(EDUCATION_MAP)
df['Education (Qualification)'].value_counts()

In [ ]:
# Adding the full plan names alongside the short codes, rather than replacing them
PLAN_TYPE_MAP = {
    'HMO': 'HMO (Health Maintenance Organization)',
    'PPO': 'PPO (Preferred Provider Organization)',
    'EPO': 'EPO (Exclusive Provider Organization)',
    'POS': 'POS (Point of Service)',
}

df['Plan Type (Full Name)'] = df['Plan Type'].map(PLAN_TYPE_MAP)
df['Plan Type (Full Name)'].value_counts()

### What these plan types actually mean

Worth writing down, since the differences drive how members use their cover:

| Plan | Full name | Needs a referral? | Covers out of network? |
|:---|:---|:---|:---|
| **HMO** | Health Maintenance Organization | Yes, through a primary care doctor | No, except emergencies |
| **PPO** | Preferred Provider Organization | No | Yes, at a higher share of cost |
| **EPO** | Exclusive Provider Organization | No | No |
| **POS** | Point of Service | Yes | Yes, at a higher share of cost |

Two other product terms appear throughout this data. The **deductible** is what a member pays out
of their own pocket each year before cover starts contributing. The **copay** is the fixed fee
they pay at each visit.

## The Numbers: Range and Spread

Now the numeric fields. I am looking for three things: values that cannot be right, such as
negative costs or ages beyond a human lifespan; the gap between the average and the middle value,
which tells me whether a few extreme members are pulling the average around; and anything sitting
at a suspicious boundary.

In [ ]:
# Summarising every numeric field
numeric_summary = df.describe().T[['mean', '50%', 'min', 'max']]
numeric_summary.columns = ['Mean', 'Median', 'Minimum', 'Maximum']
numeric_summary.round(2)

Three things stand out.

**Nothing is negative.** No impossible costs, measurements or counts.

**`Age` starts at 0**, which needs explaining — either genuine infants or a placeholder.

**`Income` is lopsided**: mean 49,874 against a median of 36,200, maximum 1,061,800. The mean
overstates a typical member's earnings, so **I use the median for income throughout**.

In [ ]:
# Looking closely at the age range, since the minimum of 0 needs explaining
df['Age'].value_counts().sort_index().head(10)

165 members recorded at age 0, then a gap with nothing at ages 1 through 4, then small numbers
from age 5 onwards.

**That gap is the tell.** In a real population of 100,000 people I would expect roughly similar
numbers of one, two, three and four-year-olds. Finding 165 members at exactly age 0 and then
nobody at all for four years means age 0 is not describing infants — it is a placeholder standing
in for something else, most likely an age that was never recorded.

## Do the Ages Match the Lives Described?

This is the check I most want to run. Each record carries an age, and separately carries
employment, marital status, education and dependents. Those should agree with one another — a
two-year-old should not be employed, married or holding a degree.

Starting with the members recorded as age 0.

In [ ]:
# Looking at what the age 0 records claim about themselves
infants = df[df['Age'] == 0]

pd.DataFrame({
    'Employment': infants['Employment Status'].value_counts(),
    'Marital status': infants['Marital Status'].value_counts(),
})

Of the 165 members recorded at age 0, 95 are marked `Employed` and a further group is marked
`Retired`. Many are recorded as married.

Taken with the missing ages 1 to 4, this confirms what the placeholder theory predicted: these
records describe adults whose age was not captured. The rest of each record describes a working,
married person — only the age field says infant.

Now let me check whether this is confined to age 0 or runs through the whole young end of the
book.

In [ ]:
# Building ten-year age bands so I can compare employment across the age range
AGE_BINS_10 = [-1, 15, 29, 39, 49, 59, 69, 120]
AGE_LABELS_10 = ['0-15 (Minors)', '16-29 (Youth)', '30-39 (Mid-30s)', '40-49 (40s)',
                 '50-59 (50s)', '60-69 (60s)', '70+ (Seniors 70+)']

df['Age Cohort (10y)'] = pd.cut(df['Age'], bins=AGE_BINS_10, labels=AGE_LABELS_10)
df['Age Cohort (10y)'].value_counts().sort_index()

In [ ]:
# Comparing employment status across every age band, as a percentage of each band
(pd.crosstab(df['Age Cohort (10y)'], df['Employment Status'], normalize='index') * 100).round(1)

Each row is an age band, each figure a percentage of that band.

If employment related to age at all, the rows would differ — under-15s barely working, over-70s
mostly retired. **Every row is the same.** `Employed` sits between 54.2% and 55.5% in every band
including 0-15; `Retired` between 19.2% and 20.5% everywhere, children included.

Employment status was assigned independently of age. It cannot be used as it stands at either end
of the range, and cleaning has to decide what to do about that. **UW**

In [ ]:
# Sizing the problem at the young end
minors = df[df['Age'] <= 17]

pd.Series({
    'Under 18 total': len(minors),
    'Recorded as working': int(minors['Employment Status'].isin(['Employed', 'Self-employed']).sum()),
    'Recorded as married': int((minors['Marital Status'] == 'Married').sum()),
    'Holding a university degree': int(minors['Education (Qualification)'].isin(['Bachelors', 'Masters', 'Doctorate']).sum()),
    'Carrying dependents': int((minors['Dependents'] > 0).sum()),
})

3,044 members are under 18: **2,072 recorded as working**, 1,632 as married, 1,330 holding a
degree, 1,761 carrying dependents.

Four fields disagree with age at once, and usually in the same record. That rules out repairing a
single field — correcting one would mean inventing a member. **Cleaning has to remove these
records rather than correct them.**

One nuance: dependents among 16-17s is plausible, marriage and doctorates among infants are not.
So the rules should tighten at the youngest bands and loosen at the oldest, not apply uniformly to
everyone under 18.

### The same question at the other end of the book

In [ ]:
# Looking at how employment is recorded among members at or past retirement age
seniors = df[df['Age'] >= 70]

pd.DataFrame({
    'Members': seniors['Employment Status'].value_counts(),
    'Share (%)': (seniors['Employment Status'].value_counts(normalize=True) * 100).round(2),
})

8,491 members are 70 or over and **4,604 — 54% — are recorded as actively employed**, against 1,742
retired.

Same artefact, opposite end, but a different problem. Here **only the employment field disagrees**.
Cost, claims, blood pressure, conditions and household are all ordinary and usable, so there is one
field to correct and 53 sound ones to keep.

That distinction drives cleaning: remove the minors, correct the field for the seniors.

## Cross-Checking the Diabetes Flag Against the Lab Result

The dataset carries two separate pieces of information about diabetes, and I can use each to
check the other.

`Diabetes` is an administrative flag — somebody has recorded that this member is diabetic.
`HbA1c Level` is a lab measurement of average blood sugar over the previous two to three months.
The clinical threshold for diagnosing diabetes is **6.5%**.

Putting them together gives four groups, and two of them are interesting.

In [ ]:
# Splitting members by what the lab says and by what the flag says
df['Glycemic Status'] = np.where(df['HbA1c Level'] >= 6.5,
                                 'HbA1c >= 6.5% (Diabetic)', 'HbA1c < 6.5% (Normal/Pre-DM)')
df['Diagnosis Status'] = np.where(df['Diabetes'] == 1, 'Diagnosed Diabetic', 'No Diagnosis Flag')

pd.crosstab(df['Diagnosis Status'], df['Glycemic Status'], margins=True)

- **91,127** normal lab, no diagnosis. Nothing to look at.
- **7,431** high lab with a diagnosis. Identified, treatment not yet controlling the level.
- **1,162** diagnosed with a normal lab. Managed back into range.
- **280** high lab and **no diagnosis at all**.

The last group is the actionable one: diabetic on their own file, invisible administratively.

In [ ]:
# Labelling the four groups so the later sections can work with them directly
DIABETES_QUADRANTS = [
    '1. Non-Diabetic / Normal',
    '2. UNDIAGNOSED DIABETIC (Silent Risk)',
    '3. Controlled Diagnosed Diabetic',
    '4. Uncontrolled Diagnosed Diabetic',
]

df['Diabetes Clinical Quadrant'] = np.select(
    [
        (df['Diabetes'] == 0) & (df['HbA1c Level'] < 6.5),
        (df['Diabetes'] == 0) & (df['HbA1c Level'] >= 6.5),
        (df['Diabetes'] == 1) & (df['HbA1c Level'] < 6.5),
        (df['Diabetes'] == 1) & (df['HbA1c Level'] >= 6.5),
    ],
    DIABETES_QUADRANTS,
    default='Unknown',
)

df.groupby('Diabetes Clinical Quadrant').agg(
    Members=('Id', 'count'),
    Mean_HbA1c=('HbA1c Level', 'mean'),
    Median_Cost=('Annual Medical Cost', 'median'),
).round(2)

The undiagnosed group averages an HbA1c of 6.62%, comfortably over the 6.5% threshold, so this is
not a handful of members sitting marginally on the line.

I am flagging this to care management rather than treating it as a data error. Nothing here is
inconsistent in the way the married infants were — the lab result and the flag are both plausible
on their own, they simply have not been reconciled with each other. **That is a gap in clinical
follow-up, not a broken record**, and no cleaning rule should touch these members. **CM**

## Why Is Alcohol Frequency Missing for 30% of Members?

Whether the gaps can be filled depends on why they are there.

If they are spread evenly across every kind of member, the cause is unrelated to the member and
filling them biases nothing. If certain groups are far more likely to be missing — heavy drinkers
declining to answer — filling them distorts the picture.

So: the missing rate, measured within every demographic group.

In [ ]:
# Flagging which members have no alcohol value recorded, and keeping that flag permanently
df['Alcohol_Missing'] = df['Alcohol Frequency'].isnull()

pd.Series({'members missing': int(df['Alcohol_Missing'].sum()),
           'share of book (%)': round(df['Alcohol_Missing'].mean() * 100, 2)})

In [ ]:
# Checking the missing rate within each demographic group, against that 30.08% baseline
pd.concat({
    feature: (df.groupby(feature)['Alcohol_Missing'].mean() * 100).round(2)
    for feature in ['Sex', 'Region', 'Urban / Rural', 'Smoker Status', 'Employment Status', 'Age Cohort (10y)']
}).rename('missing %').to_frame()

Every group sits between 29.5% and 30.4% against a 30.08% baseline — across sex, all five regions,
settlement, employment and age cohort.

Smoking is the informative one. If members were withholding drinking habits, current smokers would
differ; they do not (29.96% against 30.19% for never-smokers).

**The missingness is unrelated to anything observable**, so filling it cannot bias one part of the
book against another. What it does not explain is what the blanks *mean*.

In [ ]:
# Seeing what the field records when it is not blank
df['Alcohol Frequency'].value_counts(dropna=False)

The field takes exactly three values — `Occasional`, `Weekly`, `Daily` — and **all three describe
someone who drinks.** There is no option for a person who does not.

A question with no way to answer "I don't drink", blank for 30% of members and evenly spread, did
not fail to record answers. The blank is the answer.

Decision deferred to section 3. The `Alcohol_Missing` flag stays on the frame permanently so every
later cut can separate recorded from inferred.

## What I Found, and What Cleaning Needs to Do

**Structurally sound.** One row per member, no duplicates, no negatives, no numbers stored as text,
no inconsistent category labels. One field of 54 has missing data.

| Finding | Size | Action |
|:---|---:|:---|
| Age 0 with adult attributes, no members aged 1-4 | 165 | Remove — age 0 is a placeholder |
| Under-18s married, employed, degree-holding or with dependents | 3,044 | Remove, stricter at the youngest bands. Several fields disagree at once |
| Members 70+ recorded as employed | 4,604 | **Correct the field, keep the member.** Only employment disagrees |
| `Alcohol Frequency` blank | 30,083 | Fill as non-drinking. The survey offered no such option |

**For care management, not cleaning:** 280 members at or above the diabetic threshold with no
diagnosis flag. Their records are not broken. **CM**

**Carry forward:** employment status bears no relationship to age. Whatever cleaning does will
*create* a relationship the source did not have, and any later age-employment finding is reporting
that rule.

---

# 3. Cleaning

Section 2 changed no values. This is where the records that cannot be right get resolved.

The standard, since deleting rows is not neutral: **a record goes only when the combination of
values in it could not describe a real person.** A seven-year-old married with a doctorate is a
broken record. A 25-year-old with high blood pressure is unusual and stays.

There is a modelling reason too. Leave two thousand teenagers holding doctorates in the training
set and the model learns that education tells you nothing about age.

Where one field can be fixed instead of deleting a record, it is — once, at Rule 5.

## Keeping an Audit Trail

Before I change anything I want a record of every decision, because in six months nobody will
remember why the row count is what it is — including me. Each rule writes a line to a log, and I
keep an untouched copy of the frame so I can check at the end whether the members I removed
differed from the ones I kept.

In [ ]:
# Keeping an untouched copy, so the deleted population can be compared against the retained one
df_before_cleaning = df.copy()
deleted_ids = []

In [ ]:
# Somewhere to record every cleaning decision, so the final row count traces back to a reason
cleaning_log = []


def log_cleaning_action(step, column, action, records, rationale, impact, deleted_rows=0):
    # deleted_rows is recorded separately from Records_Affected, because a rule can touch
    # thousands of members without removing any of them. Rule 5 is exactly that case, and
    # matching on the word "deletion" in the action text would miscount it.
    cleaning_log.append({'Step': step, 'Column': column, 'Action': action,
                         'Records_Affected': records, 'Rows_Deleted': deleted_rows,
                         'Rationale': rationale, 'Downstream_Impact': impact})

## Rule 1 — Infants Recorded With Adult Lives

Starting with the members recorded as age 0, because that is where the clearest contradictions
are. An infant cannot be married, cannot have stopped smoking, cannot hold a degree, cannot be
retired, and cannot have dependents of their own.

In [ ]:
# Finding age 0 records that carry attributes an infant cannot have
is_broken_infant = (df['Age'] == 0) & (
    df['Marital Status'].isin(['Married', 'Divorced', 'Widowed'])
    | df['Smoker Status'].isin(['Current', 'Former'])
    | df['Education (Qualification)'].isin(['Primary', 'High School', 'College', 'Bachelors', 'Masters', 'Doctorate'])
    | df['Alcohol Frequency'].isin(['Occasional', 'Weekly', 'Daily'])
    | df['Employment Status'].isin(['Employed', 'Retired', 'Self-employed', 'Unemployed'])
    | (df['Dependents'] > 0)
)

pd.Series({'age 0 records in total': int((df['Age'] == 0).sum()),
           'of those, carrying an impossible attribute': int(is_broken_infant.sum())})

Both figures are 165. **Every age-0 record carries at least one impossible adult attribute** — not
most, all.

Scattered data-entry errors would hit some. Hitting all of them means age 0 is not recording
infants; it stands in for a missing or unparsed age. There is no correct value to substitute and
the rest of each record describes an adult, so all 165 go.

In [ ]:
# Removing the 165 broken infant records
deleted_ids += df.loc[is_broken_infant, 'Id'].tolist()
df = df[~is_broken_infant].copy().reset_index(drop=True)

log_cleaning_action(
    1, 'Age', 'Row deletion (corrupted age 0 records)', int(is_broken_infant.sum()),
    'Every age 0 record carried at least one impossible adult attribute, so age 0 is a placeholder '
    'for an unrecorded age rather than a real infant',
    f'Working dataset: {len(df):,} rows, ages 1 to 100',
    deleted_rows=int(is_broken_infant.sum()))

len(df)

## Building the Age Bands

Before the remaining age rules I need the bands themselves, because the rules differ by life
stage — what is impossible for a seven-year-old is ordinary for a seventeen-year-old.

The band that matters most for later is the last one. **I am setting retirement at 70**, so
anyone aged 70 or above belongs to the retiree cohort. That threshold drives Rule 5 below, and it
is a business decision rather than something the data told me.

I am building the broader life-stage grouping in the same cell, since the exploratory section
uses it to compare four coarse stages rather than nine fine ones.

In [ ]:
# Cutting age into life-stage bands, with retirement starting at 70
AGE_GROUP_BINS = [0, 7, 15, 17, 25, 35, 45, 55, 69, 120]
AGE_GROUP_LABELS = ['1-7: Preteen', '8-15: Teenager', '16-17: Preadult', '18-25: Youth',
                    '26-35: Mature Adults', '36-45: Prime Adults', '46-55: Middle-Aged Adults',
                    '56-69: Pre-Retirement Adults', '70+: Retirees']

LIFE_STAGE_BINS = [-1, 15, 29, 49, 64, 120]
LIFE_STAGE_LABELS = ['0-15 (Minors)', '16-29 (Youth)', '30-49 (Mid-Career)',
                     '50-64 (Pre-Retire)', '65+ (Seniors)']

df['Age Groups'] = pd.cut(df['Age'], bins=AGE_GROUP_BINS, labels=AGE_GROUP_LABELS)
df['Age_Life_Stage'] = pd.cut(df['Age'], bins=LIFE_STAGE_BINS, labels=LIFE_STAGE_LABELS)

log_cleaning_action(
    2, 'Age Groups / Age_Life_Stage', 'Life-stage engineering', len(df),
    'Segmented ages into nine actuarial cohorts and five broader life stages, with retirees '
    'starting at age 70',
    'Enables life-stage stratified analysis, and sets the threshold Rule 5 depends on')

pd.DataFrame({
    'Members': df['Age Groups'].value_counts().sort_index(),
    'Share (%)': (df['Age Groups'].value_counts(normalize=True).sort_index() * 100).round(2),
})

The three childhood bands are small — a few hundred to a couple of thousand members each — while
the working-age bands hold tens of thousands. That matters for the next few rules: I am about to
examine the smallest groups in the book, so a rule that removes most of a childhood band still
removes very few members overall.

## Rule 2 — Preteens Aged 1 to 7

For a one to seven-year-old I am treating these as impossible: having smoked, having been
married, drinking alcohol, holding a school or university qualification, being employed or
retired, or having dependents of their own.

One deliberate exception. I am **allowing self-employment** in this band, because a child earning
small amounts from chores or a stall is plausible enough that I would rather keep the record than
lose it.

In [ ]:
# Finding preteens carrying attributes a young child cannot have
is_broken_preteen = df['Age'].between(1, 7) & (
    df['Smoker Status'].isin(['Current', 'Former'])
    | df['Marital Status'].isin(['Married', 'Divorced', 'Widowed'])
    | df['Alcohol Frequency'].isin(['Occasional', 'Weekly', 'Daily'])
    | df['Education (Qualification)'].isin(['High School', 'College', 'Bachelors', 'Masters', 'Doctorate'])
    | df['Employment Status'].isin(['Employed', 'Retired'])
    | (df['Dependents'] > 0)
)

pd.Series({'preteens in total': int(df['Age'].between(1, 7).sum()),
           'of those, carrying an impossible attribute': int(is_broken_preteen.sum())})

472 preteens, and **all 472 match at least one impossible condition**. The same pattern as the
infants — the whole band is affected, not a scattering of it.

I am removing all of them. It costs me the entire 1-to-7 band, which is worth stating plainly:
after this step the dataset contains no young children at all. For a health insurance book that
is a real limitation, and anyone asking about paediatric cover needs to know this data cannot
answer them.

In [ ]:
# Removing the broken preteen records
deleted_ids += df.loc[is_broken_preteen, 'Id'].tolist()
df = df[~is_broken_preteen].copy().reset_index(drop=True)

log_cleaning_action(
    3, 'Age Groups (preteens)', 'Row deletion (corrupted preteen records)', int(is_broken_preteen.sum()),
    'Ages 1-7 carrying marriage, employment, retirement, degrees, dependents, smoking or drinking',
    f'Working dataset: {len(df):,} rows. No members under 8 remain, so paediatric questions '
    'cannot be answered from this book',
    deleted_rows=int(is_broken_preteen.sum()))

len(df)

## Rule 3 — Teenagers Aged 8 to 15

For this band I loosen the rules, because more becomes possible as children get older.

I still treat drinking, marriage, smoking, university-level qualifications and being employed or
retired as impossible. But I now **allow Primary and High School education**, since a
fifteen-year-old having finished primary school is entirely normal, and I **allow up to two
dependents** rather than none — a teenager could legitimately appear on a policy alongside
siblings.

The point of loosening the rules is that I want to keep every record I reasonably can. Each one
is real exposure.

In [ ]:
# Finding teenagers with attributes that do not fit the age band
is_broken_teen = df['Age'].between(8, 15) & (
    df['Alcohol Frequency'].isin(['Occasional', 'Weekly', 'Daily'])
    | df['Marital Status'].isin(['Married', 'Divorced', 'Widowed'])
    | df['Smoker Status'].isin(['Current', 'Former'])
    | df['Education (Qualification)'].isin(['College', 'Bachelors', 'Masters', 'Doctorate'])
    | df['Employment Status'].isin(['Employed', 'Retired'])
    | (df['Dependents'] > 2)
)

pd.Series({'teenagers in total': int(df['Age'].between(8, 15).sum()),
           'of those, carrying an impossible attribute': int(is_broken_teen.sum())})

1,643 teenagers, of whom 1,632 match. Eleven survive.

Loosening the rules did keep a handful of records that stricter criteria would have removed,
which is what I wanted. But eleven members is far too few to say anything about, and I suppress
the band in every later chart rather than draw conclusions from it.

In [ ]:
# Removing the teenage records that do not fit
deleted_ids += df.loc[is_broken_teen, 'Id'].tolist()
df = df[~is_broken_teen].copy().reset_index(drop=True)

log_cleaning_action(
    4, 'Age Groups (teenagers)', 'Row deletion (corrupted teenager records)', int(is_broken_teen.sum()),
    'Ages 8-15 carrying alcohol, smoking, marriage, tertiary degrees, adult employment or '
    'retirement, or more than two dependents. Primary and high school education allowed',
    f'Working dataset: {len(df):,} rows. Eleven teenagers retained, too few to report on',
    deleted_rows=int(is_broken_teen.sum()))

len(df)

## Rule 4 — Preadults Aged 16 to 17

Looser again, because sixteen and seventeen-year-olds can legitimately do most adult things.

Here I only remove records showing regular drinking, marriage or widowhood, being a **former**
smoker (current smoking is plausible at this age, having already quit is a stretch), holding a
university degree, being **retired**, or carrying more than three dependents. Employment is
allowed entirely — a seventeen-year-old with a job is ordinary.

In [ ]:
# Finding preadults with attributes that do not fit the age band
is_broken_preadult = df['Age'].between(16, 17) & (
    df['Alcohol Frequency'].isin(['Occasional', 'Weekly', 'Daily'])
    | df['Marital Status'].isin(['Married', 'Divorced', 'Widowed'])
    | (df['Smoker Status'] == 'Former')
    | df['Education (Qualification)'].isin(['College', 'Bachelors', 'Masters', 'Doctorate'])
    | (df['Employment Status'] == 'Retired')
    | (df['Dependents'] > 3)
)

pd.Series({'preadults in total': int(df['Age'].between(16, 17).sum()),
           'of those, carrying an impossible attribute': int(is_broken_preadult.sum())})

In [ ]:
# Removing the preadult records that do not fit
deleted_ids += df.loc[is_broken_preadult, 'Id'].tolist()
df = df[~is_broken_preadult].copy().reset_index(drop=True)

log_cleaning_action(
    5, 'Age Groups (preadults)', 'Row deletion (corrupted preadult records)', int(is_broken_preadult.sum()),
    'Ages 16-17 carrying alcohol, marriage, divorce or widowhood, former smoking, tertiary '
    'degrees, retirement, or more than three dependents. Employment allowed',
    f'Working dataset: {len(df):,} rows',
    deleted_rows=int(is_broken_preadult.sum()))

len(df)

764 preadults, 740 removed, 24 retained.

Taking the three childhood rules together: they removed 2,844 records and left 35 members under
18 in a book of 97,000. **This is effectively an adult-only dataset**, and I treat it as one from
here on.

## Rule 5 — Seniors Recorded as Employed

**4,604 of 8,491 seniors** are recorded as employed, 54%, against 55.4% among the under-30s.
Employment was assigned without reference to age.

### Why recode rather than delete

The childhood rules deleted because **several fields were wrong at once**, leaving no single repair
that made the record coherent.

Here **one field of 54 is wrong**. Deleting discards 53 good fields to fix one — and cuts the 70+
cohort to 3,887, **less than half**, leaving questions about older members answered by whichever
seniors the source happened not to mark as employed.

**Consequence:** `Employed` becomes 0% at 70+ by construction. That is this rule in the data, not a
finding about the portfolio.

In [ ]:
# Recoding employment for seniors, instead of deleting the records
is_working_senior = (df['Age'] >= 70) & (df['Employment Status'] == 'Employed')
n_recoded = int(is_working_senior.sum())

df.loc[is_working_senior, 'Employment Status'] = 'Retired'

log_cleaning_action(
    6, 'Employment Status (70+)', 'Field recode, not deletion', n_recoded,
    'Recoded Employed to Retired for members aged 70 and over. Retirement age is 70 and the '
    'source assigned employment independently of age (54.2% employed at 70+ vs 55.4% under 30). '
    'Only the employment field is implausible; every other attribute is valid and needed for exposure',
    f'No rows lost. Full 70+ cohort of {int((df["Age"] >= 70).sum()):,} retained. Employed is now '
    '0% at 70+ by construction and must not be reported as a finding')

df[df['Age'] >= 70]['Employment Status'].value_counts()

No seniors are marked employed any more, and the cohort still holds all 8,491 members. The row
count has not moved, which is exactly what I wanted — this rule corrected a field, it did not
remove anyone. **UW / ACT**

## Rule 6 — Missing Alcohol Frequency

Section 2 established the two facts this decision rests on. The blanks are spread evenly across
every observable group, so filling them cannot bias one part of the book against another. And the
field only ever takes three values — `Occasional`, `Weekly`, `Daily` — none of which describes a
person who does not drink.

A survey that offers no way to say "I don't drink" and then shows 30% blanks is not showing me
30% missing data. It is showing me the non-drinkers, who had nothing they could tick.

So I am filling the blanks with `Non Alcoholic`.

In [ ]:
# Filling the blanks with a category the survey never offered
n_missing_alcohol = int(df['Alcohol Frequency'].isnull().sum())
df['Alcohol Frequency'] = df['Alcohol Frequency'].fillna('Non Alcoholic')

log_cleaning_action(
    7, 'Alcohol Frequency', 'Domain-specific categorical imputation', n_missing_alcohol,
    'Filled blanks as Non Alcoholic. DISCLOSURE: this category does not exist in the recorded '
    'data, so 100% of Non Alcoholic members are imputed. Missingness is uniform across every '
    'observable segment, under which most of these members would in fact be Occasional drinkers. '
    'Alcohol_Missing flag retained so every downstream cut can separate recorded from imputed',
    'Alcohol Frequency is complete, but no underwriting or care-management decision may use it')

df['Alcohol Frequency'].value_counts()

**Disclosure — read before using any alcohol figure.** `Non Alcoholic` is synthetic. The source
recorded only Occasional, Weekly and Daily, so all 29,092 members in that category are imputed,
none observed.

The uniform missingness cuts both ways: it means filling is unbiased across groups, but it is
equally consistent with ordinary non-response, in which case most of these members drink.

An honest `Unknown` category was the alternative; it leaves 30% of the book uninterpretable on any
chart. The `Alcohol_Missing` flag is retained and 4.3 tests whether any finding depends on the
field. **No underwriting, pricing or care-management decision should use it. UW / ACT**

## Rule 7 — Young Members With Graduate Degrees

The last rule. A Masters or Doctorate takes a Bachelors first — three or four years of university
after school — and then another two to five on top. Somebody aged 18 to 21 has not had time.

In [ ]:
# Finding members too young to have completed a graduate degree
is_early_graduate = df['Age'].between(18, 21) & df['Education (Qualification)'].isin(['Masters', 'Doctorate'])

df.loc[is_early_graduate, 'Education (Qualification)'].value_counts()

352 members aged 18 to 21 hold a Masters or Doctorate.

This is the rule I am least confident in. Unlike a married infant, an advanced 21-year-old with a
Masters is rare rather than impossible. They go anyway: 352 in a four-year band is a pattern, not a
scattering of prodigies, and a model trained on it learns that age and education are unrelated.

At 0.36% of the book the cost of being wrong is small either way.

In [ ]:
# Removing the records that claim a degree the member has not had time to earn
deleted_ids += df.loc[is_early_graduate, 'Id'].tolist()
df = df[~is_early_graduate].copy().reset_index(drop=True)

log_cleaning_action(
    8, 'Education (Qualification)', 'Row deletion (implausible early graduates)', int(is_early_graduate.sum()),
    'Ages 18-21 holding Masters or Doctorate. Genuine early graduates exist, so this is '
    'implausible at scale rather than strictly impossible; at 352 in a four-year band it is '
    'indistinguishable from the independent-assignment artefact seen in the minor bands',
    f'Working dataset: {len(df):,} rows',
    deleted_rows=int(is_early_graduate.sum()))

len(df)

## Two Flags Worth Carrying Forward

Section 2 found undiagnosed diabetics by cross-referencing a lab value against a diagnosis flag,
and that column is already on the frame. The same pattern applies to blood pressure, and there is
a second oddity worth marking: members recorded as retired well before retirement age.

Neither of these removes or corrects anything. They are labels for later.

In [ ]:
# Two labels the later sections can use directly
df['undiagnosed_htn_flag'] = ((df['Systolic Blood Pressure'] >= 140) & (df['Hypertension'] == 0)).astype(int)
df['early_retired_flag'] = ((df['Age'] < 55) & (df['Employment Status'] == 'Retired')).astype(int)

log_cleaning_action(
    9, 'undiagnosed_htn_flag / early_retired_flag', 'Feature engineering', 0,
    'Marked members with a systolic reading at or above 140 and no hypertension diagnosis, and '
    'members recorded as retired before age 55. Neither changes any existing value',
    'Available to the care-management analysis in section 4')

pd.Series({'undiagnosed hypertension': int(df['undiagnosed_htn_flag'].sum()),
           'retired before 55': int(df['early_retired_flag'].sum())})

## Final Checks

Three things before I go any further: that nothing is missing, that I have not created
duplicates, and that the row count reconciles against the log.

In [ ]:
# Reconciling the row count against the log, so the arithmetic is provable
log = pd.DataFrame(cleaning_log)
deleted_total = int(log['Rows_Deleted'].sum())

pd.Series({
    'Started with': len(df_before_cleaning),
    'Rows deleted': deleted_total,
    'Rows recoded, not deleted': int(n_recoded),
    'Expected remaining': len(df_before_cleaning) - deleted_total,
    'Actually remaining': len(df),
    'Missing values anywhere': int(df.isnull().sum().sum()),
    'Duplicated rows': int(df.duplicated().sum()),
    'Unique member ids': df['Id'].nunique(),
})

Reconciles exactly: 100,000 in, 3,361 deleted, 96,639 out, nothing missing, no duplicates, grain
intact.

The log tracks rows deleted separately from records affected, and has to. Rule 5 touched 4,604
members without removing any, so counting every rule's affected records as deletions gives 7,965
and disagrees with the row count by exactly that recode.

Of the 3,361: 165 infants, 2,844 children and teenagers, 352 early graduates. **The 4,604 seniors
were recoded, not deleted.** Deleting them instead would leave 92,035 rows and a senior cohort at
less than half its real size.

In [ ]:
# The full decision log
log

## What Was Lost — Checking the Deletions for Bias

Deleting rows is not neutral, so I want to know whether the 3,361 members I removed differ from
the 96,639 I kept on the outcomes that matter. The deletions were driven by attribute
*combinations*, so similarity has to be demonstrated rather than assumed.

In [ ]:
# Comparing deleted against retained on the fields a model or an actuary cares about
was_deleted = df_before_cleaning['Id'].isin(deleted_ids)
deleted_rows = df_before_cleaning[was_deleted]
retained_rows = df_before_cleaning[~was_deleted]


def outcome_profile(frame):
    return {
        'Mean annual medical cost': frame['Annual Medical Cost'].mean(),
        'Median annual medical cost': frame['Annual Medical Cost'].median(),
        'Mean claims count': frame['Claims Count'].mean(),
        'Mean chronic conditions': frame['Chronic Conditions Count'].mean(),
        'Mean risk score': frame['Risk Score'].mean(),
        'High-risk rate (%)': frame['Is High Risk'].mean() * 100,
        'Smoker, current or former (%)': frame['Smoker Status'].isin(['Current', 'Former']).mean() * 100,
        'Female (%)': (frame['Sex'] == 'Female').mean() * 100,
        'Mean income': frame['Income'].mean(),
    }


bias_report = pd.DataFrame({
    f'Deleted (n={len(deleted_rows):,})': outcome_profile(deleted_rows),
    f'Retained (n={len(retained_rows):,})': outcome_profile(retained_rows),
}).round(2)
bias_report['Difference (%)'] = ((bias_report.iloc[:, 0] / bias_report.iloc[:, 1] - 1) * 100).round(1)
bias_report

The demographics that should not matter — sex, income, smoking — match within a few percent, so the
deletions hit the artefact rather than a real sub-population.

The outcome columns do not match. Deleted members are 23% cheaper, 15% less morbid and 66% lower
risk (a 2% high-risk rate against 38%).

That is **age structure, not bias**: the deletions removed children and young adults, `Risk Score`
is age-driven, and cost rises with age. Testing it requires comparing within one age band.

In [ ]:
# The one age band containing both deleted and retained members, so the comparison is like for like
band = df_before_cleaning[df_before_cleaning['Age'].between(18, 21)]
band_deleted = band[band['Id'].isin(deleted_ids)]
band_retained = band[~band['Id'].isin(deleted_ids)]

pd.DataFrame({
    f'Deleted 18-21 (n={len(band_deleted):,})': outcome_profile(band_deleted),
    f'Retained 18-21 (n={len(band_retained):,})': outcome_profile(band_retained),
}).round(2)

Within the 18-21 band the two groups are close on every outcome, which confirms the diagnosis: the
headline gap is age composition, not a systematic removal of one kind of member.

**Consequence to carry forward:** the cleaned book is older, sicker and more expensive than the
raw extract. Any comparison between raw and cleaned figures has to adjust for age before it means
anything. **ACT**

## Writing Out the Cleaned Data

One file leaves this notebook: the cleaned dataset. The audit script in `scripts/verify_claims.py`
reads it, so it needs to exist on disk. Everything else — the log, the bias report, the charts
below — stays in the notebook where it can be read in context.

In [ ]:
# The one dataset written to disk, because it is consumed outside this notebook
CLEAN_PATH = PROCESSED_DIR / 'medical_insurance_clean.csv'
df.to_csv(CLEAN_PATH, index=False)

df.shape

## What Came Out Of This

**96,639 members, 66 columns, no missing values, no duplicates.**

| Rule | Records | Action |
|:---|---:|:---|
| Infants with adult attributes | 165 | Deleted — every age-0 record was broken |
| Age bands | — | Nine life stages, retirement at 70 |
| Preteens aged 1-7 | 472 | Deleted — the whole band matched |
| Teenagers aged 8-15 | 1,632 | Deleted, 11 retained |
| Preadults aged 16-17 | 740 | Deleted, 24 retained |
| **Seniors marked employed** | **4,604** | **Recoded — no rows lost** |
| Missing alcohol frequency | 29,185 | Filled as Non Alcoholic |
| Under-22s with graduate degrees | 352 | Deleted |

**Three constraints on everything below:**

1. **An adult book.** 35 members under 18 survived; paediatric questions are unanswerable.
2. **`Employed` is 0% at 70+ by construction.** Any age-employment finding reports this rule.
3. **`Non Alcoholic` is 100% imputed**, inferred from absence rather than recorded.

---

# 4. Exploring the Book

Who the members are, what moves their cost, which conditions matter, and whether the products fit
the people holding them.

Underneath that sits a prior question: **can this data support those decisions at all?** Analysis
fails more often because a relationship was never there than because a chart was wrong. So each
business conclusion below is preceded by a test that the relationship exists.

## Setting Up the Charts

Everything this section needs, in one place. `matplotlib` for the plots, `scipy` for the
contingency test behind Cramér's V, `itertools` to enumerate every field pair, and `scikit-learn`
for the leakage tests at the end.

Three `scikit-learn` modules get pulled in whole here rather than split across two sections — I
need `LinearRegression`, `train_test_split` and `r2_score` below, and section 6 needs the rest of
what those same three modules offer. Importing each module exactly once, at the first point of
use, beats importing from it twice.

In [ ]:
# Adding what the exploratory work needs on top of pandas and numpy
import itertools
import warnings

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
from scipy.stats import chi2_contingency
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import (average_precision_score, brier_score_loss, classification_report,
                             mean_absolute_error, precision_score, r2_score, recall_score,
                             roc_auc_score)
from sklearn.model_selection import (KFold, StratifiedKFold, cross_val_score, learning_curve,
                                     train_test_split)

warnings.filterwarnings('ignore')

A house style, so every chart in the notebook reads the same way. The palette is colour-blind safe
and kept in a fixed order, so a given category is always the same colour wherever it appears.

The figure size is deliberately modest. I learnt that the hard way — my first pass used figures up
to 19 inches wide and every label became unreadable once the notebook scaled them down.

In [ ]:
# A colour-blind safe palette, kept in a fixed order so a category keeps its colour across charts
BLUE, ORANGE, AQUA, YELLOW = '#2a78d6', '#eb6834', '#1baf7a', '#eda100'
MAGENTA, GREEN, VIOLET, RED = '#e87ba4', '#008300', '#4a3aa7', '#e34948'
CAT = [BLUE, ORANGE, AQUA, YELLOW, MAGENTA, GREEN, VIOLET, RED]

INK, INK_2, GRID_C = '#0b0b0b', '#52514e', '#e5e4e0'

# A single-hue ramp for "how much" charts, and a blue-to-red ramp with a grey middle for
# "above or below average" charts
SEQ = LinearSegmentedColormap.from_list('seq', ['#cde2fb', '#9ec5f4', '#6da7ec', '#3987e5', '#256abf', '#184f95', '#0d366b'])
DIV = LinearSegmentedColormap.from_list('div', ['#184f95', '#3987e5', '#9ec5f4', '#f0efec', '#f2a3a2', '#e34948', '#a32b2a'])

In [ ]:
# Chart defaults, so nothing has to be restated per figure
mpl.rcParams.update({
    'figure.figsize': (10, 6),       # fits a notebook column at roughly 1:1
    'font.size': 12,                 # big enough to survive being scaled down
    'axes.titlesize': 14, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
    'axes.labelcolor': INK_2, 'axes.edgecolor': GRID_C,
    'axes.spines.top': False, 'axes.spines.right': False,
    'xtick.labelsize': 11, 'ytick.labelsize': 11,
    'xtick.color': INK_2, 'ytick.color': INK_2,
    'grid.color': GRID_C, 'legend.frameon': False,
    'figure.dpi': 100, 'savefig.dpi': 200, 'savefig.bbox': 'tight',
})

In [ ]:
# One short line per chart instead of three, since every figure goes into the written report
def save(name):
    plt.savefig(FIGS / f'{name}.png', bbox_inches='tight', pad_inches=0.3)

And the orderings, condition list and derived fields I reuse throughout. The derived fields are
analysis conveniences — a loss ratio, income quintiles, a procedure total — and section 6 drops
every one of them before it builds anything, for reasons I set out there.

In [ ]:
# Fixed orderings, so categories appear in a sensible sequence rather than alphabetically
AGE_COHORT = ['16-29 (Youth)', '30-39 (Mid-30s)', '40-49 (40s)', '50-59 (50s)',
              '60-69 (60s)', '70+ (Seniors 70+)']
LIFE_STAGE = ['16-29 (Youth)', '30-49 (Mid-Career)', '50-64 (Pre-Retire)', '65+ (Seniors)']
TIER = ['Bronze', 'Silver', 'Gold', 'Platinum']
EMPLOY = ['Employed', 'Self-employed', 'Unemployed', 'Retired']
EDU = ['Primary', 'High School', 'College', 'Bachelors', 'Masters', 'Doctorate']

In [ ]:
# The ten recorded conditions, plus shorter labels for when they need to fit on a chart axis
CONDITIONS = ['Hypertension', 'Diabetes', 'Asthma', 'COPD', 'Cardiovascular Disease',
              'Cancer History', 'Kidney Disease', 'Liver Disease', 'Arthritis',
              'Mental Health Condition']

COND_SHORT = {'Cardiovascular Disease': 'Cardiovascular', 'Cancer History': 'Cancer',
              'Mental Health Condition': 'Mental health', 'Kidney Disease': 'Kidney',
              'Liver Disease': 'Liver'}

In [ ]:
# Derived fields I need repeatedly in this section only
df['Loss Ratio'] = df['Annual Medical Cost'] / df['Annual Premium']
df['Income Band'] = pd.qcut(df['Income'], 5, labels=['Q1 lowest', 'Q2', 'Q3', 'Q4', 'Q5 highest'])
df['Total Procedures'] = df[[c for c in df.columns if c.endswith('Procedures Count')]].sum(axis=1)
df['Top Decile'] = (df['Annual Medical Cost'] > df['Annual Medical Cost'].quantile(0.9)).astype(int)

# Any group smaller than this gets suppressed rather than reported on
MIN_N = 100

---

## 4.1 Is the Premium Actually a Price?

Before I chart anything about pricing I want to check one thing. `Annual Premium` is the field a
business would naturally build a pricing story on, so if there is something wrong with it, every
conclusion after that point inherits the problem.

Starting with how strongly each field moves with medical cost.

In [ ]:
# Checking what correlates with the thing I am trying to explain
numeric = df.select_dtypes(include=[np.number]).drop(columns=['Id', 'Top Decile'])
numeric.corr()['Annual Medical Cost'].drop('Annual Medical Cost').abs().sort_values(ascending=False).head(8)

`Annual Premium` correlates with cost at **0.965**. `Total Claims Paid` at 0.74. The strongest
clinical field, chronic condition count, reaches 0.30.

That ordering is backwards. Health should predict cost; premium should be a risk judgement made in
advance. A premium tracking cost at 0.965 is not a judgement — it is arithmetic performed after the
cost was known.

Strong claim, so it gets tested directly. If premium is derived from cost, the scatter will show
lines rather than a cloud.

In [ ]:
# Taking a sample so the scatter is readable, then plotting premium against cost by tier
sample = df.sample(9000, random_state=7)

plt.figure(figsize=(10, 6))
for i, tier in enumerate(TIER):
    members = sample[sample['Network Tier'] == tier]
    plt.scatter(members['Annual Medical Cost'], members['Annual Premium'], s=6, alpha=0.45,
                color=CAT[i], label=tier, linewidths=0)

plt.xlim(0, 22000)
plt.ylim(0, 4200)
plt.xlabel('Annual medical cost')
plt.ylabel('Annual premium')
plt.title('Annual premium against annual medical cost, by network tier')
plt.legend(title='Network tier', markerscale=3)
plt.grid(alpha=0.5)
save('v2_01_premium_vs_cost')
plt.show()

##### Reading the scatter

Each point is one member: cost horizontally, premium vertically.

Risk-priced premiums would give a **cloud** — two members with the same eventual cost assessed
differently, sitting at different heights above the same point. Sloping upward, but broad.

Instead: **four straight lines**, one per tier, with almost no scatter. A straight line from near
the origin means premium is a **fixed percentage of cost**, and four lines mean that percentage
changes only with tier.

The absence of scatter is the finding. If age, conditions or claims history moved premium even
slightly, members would sit off their tier's line. Nothing about the member shifts them.

In [ ]:
# Fitting a line per tier to recover the rate each one charges per unit of cost
RATE = {tier: round(float(np.polyfit(g['Annual Medical Cost'], g['Annual Premium'], 1)[0]), 3)
        for tier, g in df.groupby('Network Tier')}

RATE

The slopes are the share of cost charged as premium: Bronze 9.6%, Silver 12%, Gold 14.4%, Platinum
17.4%.

Exact to three decimals and stepping up in order — rates from a business process are rarely this
tidy. The lines start slightly above zero, so a fixed amount is charged before cost enters. The
deductible is the only other product field that varies, which gives a full formula to test.

In [ ]:
# Testing whether a closed-form formula reproduces the premium exactly
predicted_premium = 200 + 0.01 * df['Deductible'] + df['Network Tier'].map(RATE) * df['Annual Medical Cost']
premium_error = (df['Annual Premium'] - predicted_premium).abs()

# Half a cent is just rounding, so anything within that counts as an exact match
pd.Series({'share reproduced to the cent': (premium_error <= 0.005).mean(),
           'largest error anywhere in the book': premium_error.max()})

##### The finding

**99.997%** of premiums reproduce to within half a cent — 96,636 of 96,639 — and the largest error
anywhere is **0.005**, exactly what rounding to the nearest cent produces.

```
Annual Premium = 200 + 0.01 × Deductible + tier_rate × Annual Medical Cost
tier_rate:  Bronze 0.096 · Silver 0.120 · Gold 0.144 · Platinum 0.174
```

Not an approximation of how premiums were set — the calculation itself. No member attribute enters
it.

1. **Pricing adequacy cannot be assessed.** Premium against cost measures the tier rate. Any
   loss-ratio conclusion is circular. **ACT / FIN**
2. **Premium and its relatives are barred as model features.** Enforced in section 6.
3. **Affordability remains a fair question**, since the charge ignores income entirely — 4.4.

In [ ]:
# Plotting how far each member sits from the formula prediction
plt.figure(figsize=(10, 6))
plt.hist(premium_error, bins=60, color=BLUE)
plt.xlabel('Absolute difference between actual premium and formula prediction')
plt.ylabel('Members')
plt.title('The formula reproduces every premium to within half a cent')
plt.grid(axis='y', alpha=0.5)
save('v2_02_premium_formula_error')
plt.show()

Every member sits within half a cent of the prediction and the whole distribution is bunched
inside that range. There is no tail of members the formula fails on, which is what I would expect
to see if the premium had any genuine underwriting component my formula was missing.

For completeness, the monthly premium is nothing more than the annual figure divided by twelve, so
both premium fields carry exactly the same circular information.

In [ ]:
# Confirming monthly premium carries nothing beyond annual / 12
bool(np.allclose(df['Monthly Premium'], df['Annual Premium'] / 12, atol=0.01))

---

## 4.2 What Actually Drives Cost?

Premium is circular, so the question becomes what genuinely moves cost — and therefore where
underwriting should look.

Most candidates are categorical, so correlation does not apply. I use **eta squared**: the share of
total cost variance falling between groups rather than within them, read as a percentage. 1% means
group membership accounts for 1% of why costs differ.

Eta squared drifts upward for factors with many levels, so **omega squared** is reported alongside
it as the bias-corrected counterpart.

In [ ]:
# How much of the variation in cost each grouping explains
def eta_squared(frame, cat, val='Annual Medical Cost'):
    grouped = frame.groupby(cat, observed=True)[val]
    grand_mean = frame[val].mean()
    between = (grouped.count() * (grouped.mean() - grand_mean) ** 2).sum()
    total = ((frame[val] - grand_mean) ** 2).sum()
    return between / total

In [ ]:
# The same thing, corrected for the number of groups a factor happens to have
def omega_squared(frame, cat, val='Annual Medical Cost'):
    grouped = frame.groupby(cat, observed=True)[val]
    grand_mean = frame[val].mean()
    k, n = frame[cat].nunique(), len(frame)
    ss_between = (grouped.count() * (grouped.mean() - grand_mean) ** 2).sum()
    ss_total = ((frame[val] - grand_mean) ** 2).sum()
    ms_error = (ss_total - ss_between) / (n - k)
    return max(0.0, (ss_between - (k - 1) * ms_error) / (ss_total + ms_error))

In [ ]:
# Running both across every candidate segmentation I might want to use
CANDIDATES = ['Chronic Conditions Count', 'Smoker Status', 'Had Major Procedure',
              'Age Cohort (10y)', 'Glycemic Status', 'Employment Status', 'Household Size',
              'Education (Qualification)', 'Region', 'Urban / Rural', 'Sex',
              'Marital Status', 'Income Band', 'Plan Type', 'Network Tier', 'Alcohol Frequency']

effect = pd.DataFrame({
    'eta squared %': pd.Series({c: eta_squared(df, c) * 100 for c in CANDIDATES}),
    'omega squared %': pd.Series({c: omega_squared(df, c) * 100 for c in CANDIDATES}),
}).sort_values('eta squared %', ascending=False).round(3)
effect

The two columns agree on everything that matters. The same four variables lead on both measures
and every demographic stays at or below roughly 0.1% either way. With 96,639 members the bias was
never going to be the risk; running the correction simply removes the doubt.

There is a very sharp cliff in this list. Four variables clear 1%, and everything below them falls
away to fractions of a percent. Worth charting, because reading down a column makes 8.76 and 0.09
look like neighbours when one is roughly a hundred times the other.

In [ ]:
# Plotting the effect sizes, colouring anything under 1% differently since it is not usable
league = effect['eta squared %']

plt.figure(figsize=(10, 6))
colours = [BLUE if v >= 1 else ORANGE for v in league.values][::-1]
bars = plt.barh(range(len(league)), league.values[::-1], color=colours)
plt.yticks(range(len(league)), league.index[::-1])

for bar, value in zip(bars, league.values[::-1]):
    plt.annotate(f'{value:.2f}%', (bar.get_width(), bar.get_y() + bar.get_height() / 2),
                 xytext=(4, 0), textcoords='offset points', va='center', fontsize=10, color=INK_2)

plt.axvline(1, color='grey', ls='--')
plt.xlabel('Share of cost variation explained (%)')
plt.title('Only clinical burden, smoking, procedures and age move cost')
plt.xlim(0, 11)
plt.grid(axis='x', alpha=0.5)
save('v2_03_cost_drivers')
plt.show()

##### Interpretation

Four variables clear 1%: chronic condition count **8.8%**, smoking 2.6%, major procedure 2.2%, age
cohort 1.6%.

8.8% is the strongest thing here and still leaves 91% unexplained — normal for individual medical
cost, which is driven heavily by chance. Best available segmentation; no basis for predicting an
individual.

Below the line is almost everything a business would segment on. Employment **0.09%**, region
**0.008%** — a member's region leaves you no better placed to guess their cost than knowing nothing.
Income, sex, marital status, education, household size and plan type are effectively zero.

Region is not useless, it is not a *risk* factor: where members are, not what they cost. **UW / ACT**

One to carry: smoking is second strongest here, yet 4.5 shows smokers with near-identical disease
rates.

### Do the drivers stack, or are they the same thing measured four ways?

The league table above measures each variable on its own. That leaves an obvious objection: if the
four leaders are all capturing the same underlying thing, adding them together would explain no
more than the best of them alone, and the "demography is inert" verdict might be an artefact of
testing one variable at a time.

So I fit two models on log cost. The first uses only the four clinical and utilisation drivers;
the second adds every demographic field in the book at once.

In [ ]:
# Fitting on log cost, because the raw amounts are lopsided enough to distort a straight-line fit
log_cost = np.log1p(df['Annual Medical Cost'])

DRIVER_COLS = ['Age', 'Chronic Conditions Count', 'Smoker Status',
               'Hospitalizations in Last 3 Years', 'Days Hospitalized in Last 3 Years']
DEMOGRAPHIC_COLS = ['Region', 'Urban / Rural', 'Sex', 'Education (Qualification)',
                    'Marital Status', 'Employment Status', 'Household Size', 'Income']


def linear_r2(columns):
    encoded = pd.get_dummies(df[columns], drop_first=True)
    X_tr, X_te, y_tr, y_te = train_test_split(encoded, log_cost, test_size=0.3, random_state=42)
    return r2_score(y_te, LinearRegression().fit(X_tr, y_tr).predict(X_te))


driver_r2 = linear_r2(DRIVER_COLS)
full_r2 = linear_r2(DRIVER_COLS + DEMOGRAPHIC_COLS)

pd.Series({'Four clinical and utilisation drivers': round(driver_r2, 3),
           'The same four plus every demographic': round(full_r2, 3),
           'What the demographics add': round(full_r2 - driver_r2, 3)})

The four clinical and utilisation drivers jointly account for roughly 16% of the variation in log
cost. Adding every demographic field at once moves that by less than a point.

So the verdict holds when the variables are tested together, not just one at a time. And that
**16% is the honest ceiling** any cost model on this book should be judged against — I come back
to it in section 6 when a model scores 0.17 and somebody wants to know whether that is good.

---

## 4.3 Testing Every Relationship

Everything so far asked one question: does X move cost? That leaves a gap I am not comfortable
with, because it never tests whether variables relate to *each other*.

Different data types need different statistics, so I need three, all reporting on the same 0-to-1
scale: **Cramér's V** for two categories, the **correlation ratio** for a category against a
number, and ordinary correlation for two numbers.

In [ ]:
# Cramer's V for two categorical fields, corrected for the bias small tables introduce
def cramers_v(a, b):
    table = pd.crosstab(a, b)
    if table.shape[0] < 2 or table.shape[1] < 2:
        return np.nan
    chi2 = chi2_contingency(table)[0]
    n = table.values.sum()
    phi2 = chi2 / n
    r, k = table.shape
    phi2_corrected = max(0, phi2 - (k - 1) * (r - 1) / (n - 1))
    r_corrected = r - ((r - 1) ** 2) / (n - 1)
    k_corrected = k - ((k - 1) ** 2) / (n - 1)
    return np.sqrt(phi2_corrected / max(1e-12, min(k_corrected - 1, r_corrected - 1)))

In [ ]:
# The correlation ratio, for a categorical field against a numeric one
def correlation_ratio(cat, num):
    grouped = num.groupby(cat, observed=True)
    grand_mean = num.mean()
    between = (grouped.count() * (grouped.mean() - grand_mean) ** 2).sum()
    total = ((num - grand_mean) ** 2).sum()
    return np.sqrt(between / total) if total > 0 else np.nan

In [ ]:
# Picking the right statistic automatically from the two field types.
# Testing the dtype by name rather than against `object`, because pandas 3 stores text as
# `str` and an `== object` check silently returns False for every text column.
def is_categorical(col):
    return str(df[col].dtype) in ('str', 'object', 'category') or df[col].nunique() <= 10


def association(a, b):
    cat_a, cat_b = is_categorical(a), is_categorical(b)
    if cat_a and cat_b:
        return cramers_v(df[a], df[b]), 'Cramers V'
    if cat_a:
        return correlation_ratio(df[a], df[b]), 'eta'
    if cat_b:
        return correlation_ratio(df[b], df[a]), 'eta'
    return abs(df[a].corr(df[b])), '|r|'

One thing has to be excluded first. Several columns are re-encodings of columns I already have —
`Age Cohort (10y)` is just `Age` in bands, `Income Band` is `Income` in quintiles. Testing `Age`
against `Age Cohort` would measure my own binning rather than the portfolio, and those tautologies
would dominate the top of the results.

In [ ]:
# Dropping re-encodings of columns I already have, so I do not measure my own binning
DERIVED = ['Id', 'Plan Type (Full Name)', 'Age Groups', 'Age Cohort (10y)', 'Age_Life_Stage',
           'Glycemic Status', 'Diagnosis Status', 'Diabetes Clinical Quadrant',
           'Alcohol_Missing', 'Income Band', 'Top Decile', 'Loss Ratio',
           'Total Procedures', 'Monthly Premium', 'undiagnosed_htn_flag', 'early_retired_flag']

SCAN = [c for c in df.columns if c not in DERIVED]
len(SCAN)

In [ ]:
# Testing every possible pair of the remaining fields
pairs = []
for a, b in itertools.combinations(SCAN, 2):
    value, stat = association(a, b)
    if value == value:                          # skipping any pair that comes back NaN
        pairs.append((a, b, stat, value))

scan = pd.DataFrame(pairs, columns=['A', 'B', 'stat', 'value']).sort_values('value', ascending=False)
len(scan)

1,326 pairs tested — every field against every other field.

Before looking at any individual pair I want the overall picture, so I am sorting them into bands.
All three measures run on the same 0-to-1 scale, where 0 means the two fields tell you nothing
about each other and 1 means one determines the other completely. The conventional reading is that
anything under 0.05 is noise, 0.10 to 0.15 is weak, 0.15 to 0.30 is moderate, and above 0.30 is
strong.

In [ ]:
# Grouping the results into strength bands to see the overall picture
bands = pd.cut(scan.value, [-0.001, 0.05, 0.10, 0.15, 0.30, 1.001],
               labels=['under 0.05 (none)', '0.05-0.10 (trivial)', '0.10-0.15 (weak)',
                       '0.15-0.30 (modest)', 'over 0.30 (strong)'])
bands.value_counts().sort_index()

**988 of the 1,326 pairs — 75% of everything tested — come in under 0.05.** Three quarters of the
field pairs in this dataset tell you nothing whatsoever about each other. Only 47 pairs, about
3.5%, reach the 0.30 mark that counts as strong.

That is a sparse dataset. Most of what could relate to something else simply does not. Now let me
look at what those strong pairs actually are, because a strong number is not automatically an
interesting finding.

In [ ]:
# The strongest relationships anywhere in the dataset
scan.head(12).reset_index(drop=True)

Almost every strong pair is two views of one thing rather than two things that move together:

- `Annual Medical Cost` ↔ `Annual Premium`, 0.965 — the formula from 4.1
- `Hospitalizations` ↔ `Days Hospitalized`, 0.89 — days follow admissions
- `Surgical Procedures` ↔ `Had Major Procedure`, 0.84 — the flag is derived from the count
- `Risk Score` ↔ `Is High Risk`, 0.83 — a thresholded version of the score
- `HbA1c Level` ↔ `Diabetes`, 0.79 — HbA1c is the diagnostic

The only pair describing two separate things is `Household Size` ↔ `Dependents` at 0.76.

Ranking by strength is therefore misleading, and the pairs need classifying by kind.

In [ ]:
# Splitting the relationships by whether they are genuine or just circular
FINANCIAL = ['Annual Premium', 'Total Claims Paid', 'Average Claim Amount',
             'Annual Medical Cost', 'Risk Score', 'Is High Risk']

# Pairs where one field is computed from, or clinically defines, the other
DEFINITIONAL = [
    ({'Chronic Conditions Count'}, set(CONDITIONS)),
    ({'Hypertension'}, {'Systolic Blood Pressure', 'Diastolic Blood Pressure'}),
    ({'Diabetes'}, {'HbA1c Level'}),
    ({'Had Major Procedure'}, {'Surgical Procedures Count', 'Hospitalizations in Last 3 Years',
                               'Days Hospitalized in Last 3 Years'}),
    ({'Hospitalizations in Last 3 Years'}, {'Days Hospitalized in Last 3 Years'}),
    ({'Claims Count'}, {'Visits in Last Year'}),
]


def classify(a, b):
    if a in FINANCIAL and b in FINANCIAL:
        return 'financial (computed from each other)'
    for left, right in DEFINITIONAL:
        if (a in left and b in right) or (b in left and a in right):
            return 'definitional (a field and its own parts)'
    if a in FINANCIAL or b in FINANCIAL:
        return 'clinical or utilisation driving cost'
    return 'structural (a real relationship)'

In [ ]:
# Classifying everything above the weak threshold
strong = scan[scan.value >= 0.10].copy()
strong['kind'] = [classify(a, b) for a, b in zip(strong.A, strong.B)]
strong.kind.value_counts()

**64 of 1,326 pairs — under 5% — describe a genuine connection between separately measured things.**
The rest is arithmetic, definition, or one financial field against another computed from it.

Those 64 fall into three groups: family composition, the body changing with age, and members with
more conditions using more care.

Nothing connects member demographics to product held. That absence closes several business
questions at once — 4.9.

### Putting specific business hypotheses to the test

The scan is thorough but abstract. What a reviewer will actually want to know is whether the
specific things they believe are true, so I wrote down the hypotheses somebody would reasonably
raise about a health book and measured each one.

In [ ]:
# The hypotheses a business would actually ask about, written down to be tested rather than assumed
HYPOTHESES = [
    ('Smokers are more likely to have dependents', 'Smoker Status', 'Dependents'),
    ('Smokers have a higher BMI', 'Smoker Status', 'BMI'),
    ('Higher earners buy the richer tiers', 'Income Band', 'Network Tier'),
    ('Bigger households buy richer cover', 'Household Size', 'Network Tier'),
    ('The unemployed buy cheaper cover', 'Employment Status', 'Network Tier'),
    ('Rural members consume less care', 'Urban / Rural', 'Visits in Last Year'),
    ('Rural members cost less', 'Urban / Rural', 'Annual Medical Cost'),
    ('Better educated members are healthier', 'Education (Qualification)', 'Chronic Conditions Count'),
    ('Higher earners are healthier', 'Income Band', 'Chronic Conditions Count'),
    ('Married members claim more', 'Marital Status', 'Claims Count'),
    ('Diabetics choose lower deductibles', 'Diabetes', 'Deductible'),
    ('The chronically ill choose richer networks', 'Chronic Conditions Count', 'Network Tier'),
    ('Men and women differ in morbidity', 'Sex', 'Chronic Conditions Count'),
    ('Older members hold more conditions', 'Age', 'Chronic Conditions Count'),
    ('Household size tracks dependents', 'Household Size', 'Dependents'),
    ('Prior hospitalisation predicts cost', 'Hospitalizations in Last 3 Years', 'Annual Medical Cost'),
]

In [ ]:
# Measuring each one against the same thresholds used above
verdicts = []
for label, a, b in HYPOTHESES:
    value, stat = association(a, b)
    verdict = ('structural' if value >= 0.30 else
               'supported' if value >= 0.15 else
               'weak' if value >= 0.10 else 'not supported')
    verdicts.append({'hypothesis': label, 'statistic': stat, 'value': round(value, 4), 'verdict': verdict})

pd.DataFrame(verdicts).sort_values('value', ascending=False).reset_index(drop=True)

##### What this table says

Almost every hypothesis a business would raise fails.

Two clear the bar. Household size tracks dependents at 0.76, which is one fact stated twice. Prior
hospitalisation predicts cost at 0.21 — moderate, actionable, with individual exceptions. That one
is built on in 4.10.

Smokers and dependents, the question that prompted the exercise: **0.000**.

The product hypotheses fail hardest. Higher earners buying richer tiers 0.003; bigger households
0.001; the unemployed buying cheaper cover 0.000 exactly. **Nothing about a member predicts which
product they hold.** **PROD / ACT**

The value of testing all 1,326 pairs is being able to say each of these came back empty, rather
than that it never occurred to me to look.

### Seeing the whole picture at once

A table of 1,326 rows is impossible to read as a whole, but a heatmap of the same numbers is
immediate. I am grouping the fields into families so the block structure is visible.

In [ ]:
# Grouping fields into families so related blocks sit together on the heatmap
BLOCKS = {
    'Demographic': ['Age', 'Sex', 'Region', 'Urban / Rural', 'Income',
                    'Education (Qualification)', 'Marital Status', 'Employment Status',
                    'Household Size', 'Dependents'],
    'Lifestyle': ['BMI', 'Smoker Status', 'Alcohol Frequency'],
    'Clinical': CONDITIONS + ['Chronic Conditions Count', 'Systolic Blood Pressure',
                              'Diastolic Blood Pressure', 'LDL Cholesterol', 'HbA1c Level'],
    'Utilisation': ['Visits in Last Year', 'Hospitalizations in Last 3 Years',
                    'Days Hospitalized in Last 3 Years', 'Medication Count', 'Claims Count'],
    'Product': ['Plan Type', 'Network Tier', 'Deductible', 'Copay',
                'Policy Term (Years)', 'Policy Changes in Last 2 Years'],
    'Financial': ['Annual Medical Cost', 'Annual Premium', 'Total Claims Paid',
                  'Average Claim Amount', 'Risk Score'],
}

ORDER, boundaries, block_labels = [], [], []
for block, cols in BLOCKS.items():
    cols = [c for c in cols if c in SCAN]
    ORDER += cols
    boundaries.append(len(ORDER))
    block_labels.append((len(ORDER) - len(cols) / 2, block))

len(ORDER)

In [ ]:
# Building the square matrix from the pair list
matrix = pd.DataFrame(np.nan, index=ORDER, columns=ORDER)
for a, b, stat, value in scan.itertuples(index=False):
    if a in matrix.index and b in matrix.index:
        matrix.loc[a, b] = value
        matrix.loc[b, a] = value

matrix.shape

In [ ]:
# Drawing every tested relationship as one picture
plt.figure(figsize=(12, 11))
plt.imshow(matrix.values, cmap=SEQ, vmin=0, vmax=0.6)
plt.xticks(range(len(ORDER)), ORDER, rotation=90, fontsize=8)
plt.yticks(range(len(ORDER)), ORDER, fontsize=8)

# Marking where one family of fields ends and the next begins
for bound in boundaries[:-1]:
    plt.axhline(bound - 0.5, color=RED, lw=1.4)
    plt.axvline(bound - 0.5, color=RED, lw=1.4)
for pos, label in block_labels:
    plt.annotate(label, (pos - 0.5, -3), ha='center', fontsize=11, fontweight='bold', color=RED)

colourbar = plt.colorbar(fraction=0.03, pad=0.02)
colourbar.set_label('Strength of relationship')
plt.title('Every field against every other field, all 1,326 pairs')
save('v2_04_association_matrix')
plt.show()

##### Reading the heatmap

Each square is one pair; darker means stronger. Red lines separate the families.

The picture is **pale almost everywhere**. Darkness is confined to three patches: family
composition in the demographic block; the join between clinical and utilisation, where more
conditions means more visits, medication and procedures; and the financial block, dark because
those fields are computed from one another.

**The Product row and column are blank.** Not faint — blank. Nothing demographic, clinical or
financial relates to plan type, tier, deductible or copay.

One visible artefact that is not a finding: the mild `Age` ↔ `Employment Status` link is the
cleaning rule recoding over-70s to `Retired`. It is excluded from the count of 64.

### Does the alcohol imputation change anything?

Section 3 disclosed that all 29,092 `Non Alcoholic` members are imputed rather than observed. That
is a large enough assumption that I would rather test its consequences than carry it quietly, so:
do the recorded and imputed members differ, and does the field carry any signal either way?

In [ ]:
# Comparing recorded against imputed members, then measuring the field's own signal both ways
recorded = df[~df['Alcohol_Missing']]
imputed = df[df['Alcohol_Missing']]


def alcohol_profile(frame):
    return {
        'Mean annual cost': frame['Annual Medical Cost'].mean(),
        'In the top cost decile (%)': frame['Top Decile'].mean() * 100,
        'Mean chronic conditions': frame['Chronic Conditions Count'].mean(),
        'Current smoker (%)': (frame['Smoker Status'] == 'Current').mean() * 100,
    }


pd.DataFrame({
    f'Recorded (n={len(recorded):,})': alcohol_profile(recorded),
    f'Imputed (n={len(imputed):,})': alcohol_profile(imputed),
}).round(2)

In [ ]:
# And how much cost variation the field explains, with and without the imputed group
pd.Series({
    'eta squared %, imputed group included': round(eta_squared(df, 'Alcohol Frequency') * 100, 3),
    'eta squared %, recorded values only': round(eta_squared(recorded, 'Alcohol Frequency') * 100, 3),
})

Recorded and imputed members are statistically indistinguishable on cost, morbidity and smoking,
and the field explains close to nothing about cost whether the imputed group is included or not.

So no finding in this notebook depends on the alcohol field. That is a relief rather than a
vindication — the disclosure from section 3 still stands, and the field should not feed any
underwriting or care-management decision. **UW**

---

## 4.4 Who Is In This Book?

Section 4.2 showed none of these demographic cuts move cost, so I am not going to present them as
risk factors. But they still tell me who the members are, which matters for distribution and for
understanding the shape of the portfolio. I am reading them as **exposure, not as risk**.

In [ ]:
# The age profile, and where the exposure actually concentrates
pd.concat([
    df['Age'].describe().round(1).rename('value').to_frame(),
    (df['Age Cohort (10y)'].value_counts(normalize=True).reindex(AGE_COHORT) * 100).round(1).rename('value').to_frame(),
])

A middle-aged book. Median age 48, running from 16 to 100. The 40s, 50s and 60s carry most of it —
about 63% of members between 40 and 69 — with a 10% youth cohort and just under 9% over 70. This is
a working-age book with a meaningful senior tail.

### Where members are, geographically

In [ ]:
# How the book splits across regions
region_share = (df['Region'].value_counts(normalize=True) * 100).round(1)

plt.figure(figsize=(10, 5))
bars = plt.barh(range(len(region_share)), region_share.values[::-1], color=BLUE)
plt.yticks(range(len(region_share)), region_share.index[::-1])

for bar, value in zip(bars, region_share.values[::-1]):
    plt.annotate(f'{value}%', (bar.get_width(), bar.get_y() + bar.get_height() / 2),
                 xytext=(4, 0), textcoords='offset points', va='center', fontsize=11, color=INK_2)

plt.xlabel('Share of the book (%)')
plt.title('South carries the most exposure, Central the least')
plt.xlim(0, 34)
plt.grid(axis='x', alpha=0.5)
save('v2_05_region_exposure')
plt.show()

South is the largest region at 28% of the book and Central the smallest at 12.1% — more than a
twofold difference in how much business each region carries. That is a real distribution fact and
it tells the sales side something about where the book is concentrated.

Before anyone reads a risk story into it, though, the regions need checking for what they cost.

In [ ]:
# Checking whether the regions actually differ in cost or morbidity
df.groupby('Region').agg(
    members=('Id', 'size'),
    median_cost=('Annual Medical Cost', 'median'),
    mean_conditions=('Chronic Conditions Count', 'mean'),
).round(2)

The regions are essentially identical. Median cost spans about 46 across all five — on a median of
roughly 2,100, that is a 2% spread — and average chronic conditions differ in the second decimal
place.

So region tells you **where to sell, not what to charge**. A regional pricing or underwriting
strategy has nothing to stand on in this data, and I would not fund a regional risk initiative
from it. **NET / ACT**

### Employment, and the insured unemployed

A meaningful number of unemployed people hold cover here, which is interesting commercially — if
you can work out why they buy, you can sell to more of them.

In [ ]:
# Comparing the unemployed cohort against the book average on everything I can measure
METRICS = {'Median age': ('Age', 'median'), 'Median income': ('Income', 'median'),
           'Mean conditions': ('Chronic Conditions Count', 'mean'),
           'Median cost': ('Annual Medical Cost', 'median'),
           'Mean risk score': ('Risk Score', 'mean'),
           'Mean visits': ('Visits in Last Year', 'mean')}

unemployed = df[df['Employment Status'] == 'Unemployed']

pd.DataFrame({
    'Members': df['Employment Status'].value_counts(),
    'Share (%)': (df['Employment Status'].value_counts(normalize=True) * 100).round(1),
})

In [ ]:
# Indexing the unemployed cohort against the book, to see whether it differs on anything at all
pd.Series({label: round((unemployed[col].agg(how) / df[col].agg(how) - 1) * 100, 1)
           for label, (col, how) in METRICS.items()},
          name='Difference from book average (%)')

12,521 unemployed members hold cover, 13% of the book, and they sit within a few percent of the
book average on **every dimension measured** — age, income, morbidity, cost, utilisation.

A disappointing answer to a good question. Nothing recorded here explains why an unemployed person
buys cover; that needs acquisition data — channel, prior cover, purchase trigger — which is not in
this extract.

Better said plainly than dressed up as a 2% difference that is almost certainly noise.

(Retired members are 24.6%, inflated by the Rule 5 recode, so that figure is not read as a finding.)

### Can members afford this?

Section 4.1 established that premium is calculated from realised cost and ignores income
completely. That makes affordability a fair question: if the charge takes no account of what you
earn, who does it fall hardest on?

In [ ]:
# Comparing what each income quintile earns against what they are charged
df.groupby('Income Band', observed=True).agg(
    members=('Id', 'size'),
    median_income=('Income', 'median'),
    median_premium=('Annual Premium', 'median'),
).round(0)

This is the whole finding in one table. Median income runs from 13,000 in the bottom quintile to
101,200 in the top — a **7.8 times** difference. Median premium runs from 467 to 466. It is
completely **flat**.

The consequence is arithmetic. Let me measure the burden per member rather than by dividing the
medians, because income is skewed within each quintile too.

In [ ]:
# What share of income each member actually pays, taken as a median within each quintile
df['Burden %'] = df['Annual Premium'] / df['Income'] * 100
burden = df.groupby('Income Band', observed=True)['Burden %'].median()

plt.figure(figsize=(10, 6))
bars = plt.bar(range(len(burden)), burden.values, color=RED, width=0.6)
plt.xticks(range(len(burden)), burden.index, rotation=15)

for bar, value in zip(bars, burden.values):
    plt.annotate(f'{value:.2f}%', (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                 xytext=(0, 4), textcoords='offset points', ha='center', fontsize=11, color=INK_2)

plt.ylabel('Premium as a share of income (%)')
plt.title('The poorest quintile pays 8.8 times the share of income the richest pays')
plt.grid(axis='y', alpha=0.5)
save('v2_06_affordability_burden')
plt.show()

##### What the burden chart shows

The poorest quintile pays **3.96% of income**; the richest **0.45%**. An 8.8-fold difference in
burden for essentially the same charge.

The mechanism is the 4.1 formula: premium keys off realised cost and tier, and income never enters
it.

Whether that is acceptable is policy, not analysis — some contribution schemes are deliberately
flat. It should be a deliberate choice, and product and compliance should know the structure has
this shape. **PROD / FIN**

---

## 4.5 What Does Demography Tell Us About Health?

This is the section I was most interested in going in. If the demographic attributes we collect
tell us anything about a member's health, underwriting can use them.

I am testing all ten recorded conditions against every demographic cut, rather than picking a
couple of conditions and a couple of cuts. I want the pattern, not an anecdote.

In [ ]:
# How common each condition is across the whole book
(df[CONDITIONS].mean() * 100).sort_values(ascending=False).round(1)

Hypertension is by far the most common at 20.4%, then mental health at 13.1% and arthritis at
10.9%. Kidney and liver disease are rare at around 1.5% each.

Now the real question: does any demographic cut change these numbers? I am building a grid — every
condition against every demographic — and colouring by how far each group sits from the portfolio
baseline for that condition. If demography stratifies health, blocks of this grid should light up.

In [ ]:
# The demographic cuts to test each condition against
CUTS = [('Age Cohort (10y)', AGE_COHORT),
        ('Urban / Rural', ['Urban', 'Suburban', 'Rural']),
        ('Region', ['North', 'South', 'East', 'West', 'Central']),
        ('Employment Status', EMPLOY),
        ('Sex', ['Female', 'Male']),
        ('Education (Qualification)', EDU),
        ('Marital Status', ['Single', 'Married', 'Divorced', 'Widowed']),
        ('Smoker Status', ['Never', 'Former', 'Current']),
        ('Income Band', list(df['Income Band'].cat.categories))]

# Dropping the handful of retained minors, who are too few to read
adults = df[df['Age_Life_Stage'] != '0-15 (Minors)']
baseline = adults[CONDITIONS].mean() * 100

len(adults)

In [ ]:
# One panel per demographic, ten conditions down the side, coloured by distance from baseline
fig, axes = plt.subplots(1, len(CUTS), figsize=(18, 6),
                         gridspec_kw={'width_ratios': [len(o) for _, o in CUTS], 'wspace': 0.1})
row_labels = [COND_SHORT.get(c, c) for c in CONDITIONS]

for ax, (cut, order) in zip(axes, CUTS):
    order = [o for o in order if o in adults[cut].astype(str).unique()]
    rates = (adults.groupby(cut, observed=True)[CONDITIONS].mean() * 100).reindex(order)
    counts = adults.groupby(cut, observed=True).size().reindex(order)
    relative = rates.sub(baseline, axis=1).div(baseline, axis=1) * 100
    relative = relative.mask(counts.lt(MIN_N), np.nan)          # suppressing thin groups

    im = ax.imshow(relative.T.values, cmap=DIV, aspect='auto',
                   norm=TwoSlopeNorm(vmin=-45, vcenter=0, vmax=45))
    ax.set_xticks(range(len(order)),
                  [str(o).split(' (')[0].replace(' lowest', '').replace(' highest', '') for o in order],
                  rotation=90, fontsize=9)
    ax.set_yticks(range(len(row_labels)), row_labels if ax is axes[0] else [''] * len(row_labels), fontsize=10)
    ax.set_title(cut.split(' (')[0], fontsize=10)
    for spine in ax.spines.values():
        spine.set_visible(False)

colourbar = fig.colorbar(im, ax=axes, fraction=0.012, pad=0.012)
colourbar.set_label("Difference from the condition's baseline (%)")
fig.suptitle('Ten conditions against every demographic cut', x=0.09, ha='left',
             fontsize=14, fontweight='bold', y=1.02)
save('v2_07_morbidity_grid')
plt.show()

##### Reading the grid

Rows are conditions, columns are groups, colour is distance from that condition's book-wide rate.
**Red above, blue below, pale means matching the book.**

Only the age panel shows a gradient — blue young, red old, on almost every condition. Settlement,
region, employment, sex, education, marital status and income band are all pale: each group within
a few percent of the book average on all ten conditions.

**Smoking is the exception worth stating.** Smokers show never-smoker rates across all ten,
including COPD, cardiovascular disease and cancer — yet 4.2 found smoking the second strongest cost
driver. Higher spending, no higher recorded disease. Both reported.

**Age is the only demographic worth using to stratify health risk. UW**

In [ ]:
# Plotting all ten conditions across the age cohorts, highlighting the ones that actually climb
by_age = (adults.groupby('Age Cohort (10y)', observed=True)[CONDITIONS].mean() * 100).reindex(AGE_COHORT)

plt.figure(figsize=(10, 6))
x = range(len(AGE_COHORT))

for cond in CONDITIONS:
    values = by_age[cond].values
    steep = values[-1] / values[0] >= 1.25
    plt.plot(x, values, marker='o', markersize=5,
             color=BLUE if steep else '#c9c8c3', lw=2.5 if steep else 1.5, zorder=3 if steep else 1)
    if steep:
        plt.annotate(f'{COND_SHORT.get(cond, cond)}  {values[-1]:.0f}%', (len(x) - 1, values[-1]),
                     xytext=(6, 0), textcoords='offset points', va='center',
                     fontsize=10, color=BLUE, fontweight='bold')

plt.xticks(x, [c.split(' (')[0] for c in AGE_COHORT], rotation=20)
plt.xlabel('Age cohort')
plt.ylabel('Prevalence (%)')
plt.title('Conditions that climb with age (blue) against those that do not (grey)')
plt.xlim(-0.2, len(x) + 1.1)
plt.grid(alpha=0.5)
save('v2_08_age_morbidity_gradient')
plt.show()

Six of ten conditions rise with age. Hypertension 16.8% to 26.2%, arthritis 8.7% to 14.7%, mental
health 11.1% to 17.3%, diabetes 7.2% to 10.8%. The four grey lines — cancer, COPD, liver, kidney —
are recorded at much the same rate at both ends.

Size matters as much as direction. A 70-year-old is roughly **1.5 times** as likely to have
hypertension as someone in their twenties, not five or ten times: about one in six of the youngest
cohort against one in four of the oldest. Most older members still do not have it.

Which matches the 1.6% from 4.2. Age is usable and it is the only demographic stratifier available,
but it is a weak one.

### Do conditions cluster together?

In real populations, diseases co-occur. Diabetics develop kidney disease. People with COPD develop
cardiovascular problems. If that clustering exists here, then multi-morbid members are a
qualitatively different risk rather than just an arithmetic sum.

In [ ]:
# How much more likely each condition is, given a member already has another one.
# A lift of 1.0 means the two conditions are completely independent of each other.
lift = np.zeros((len(CONDITIONS), len(CONDITIONS)))
for i, a in enumerate(CONDITIONS):
    for j, b in enumerate(CONDITIONS):
        lift[i, j] = np.nan if i == j else df.loc[df[a] == 1, b].mean() / df[b].mean()

pd.DataFrame(lift,
             index=[COND_SHORT.get(c, c) for c in CONDITIONS],
             columns=[COND_SHORT.get(c, c) for c in CONDITIONS]).round(2)

Each figure is a **lift** — how much more likely one condition is given another, against a member
picked at random. 1.00 means knowing the first tells you nothing about the second.

Every off-diagonal value sits at almost exactly 1.00. Diabetes does not raise kidney disease risk;
COPD does not raise cardiovascular risk. **The conditions occur independently.**

So multi-morbid members cost more because each condition adds its own cost, not because the
combination compounds. Size the multi-morbid group expecting cost to add rather than accelerate,
and do not price a fourth condition as more dangerous than the third. **ACT / CM**

### The silent risk: diabetics nobody has diagnosed

Section 2 built a field cross-referencing each member's HbA1c reading against whether they carry a
diabetes flag. Anyone at or above the 6.5% diagnostic threshold without a flag is diabetic on
paper but invisible administratively.

In [ ]:
# How members split across the diabetes quadrants
undiagnosed = df['Diabetes Clinical Quadrant'].str.contains('UNDIAGNOSED')

pd.DataFrame({
    'Members': df['Diabetes Clinical Quadrant'].value_counts().sort_index(),
    'Share of book (%)': (df['Diabetes Clinical Quadrant'].value_counts(normalize=True).sort_index() * 100).round(2),
    'Share of spend (%)': (df.groupby('Diabetes Clinical Quadrant')['Annual Medical Cost'].sum()
                           / df['Annual Medical Cost'].sum() * 100).round(2),
})

**273 members — 0.28% of the book — sit at or above HbA1c 6.5% with no diabetes flag.** Diabetic on
their own file, invisible administratively.

Sized honestly: 0.28% of members, **0.3% of spend**. This does not move the portfolio's financial
position.

It is worth doing because it costs almost nothing. The lab value is already held, so identifying
them is a query, not a screening programme. A clinical-governance item with a list attached — not a
cost-saving initiative. **CM**

---

## 4.6 Where Does The Money Go?

Now to follow the money. This is the part with the most genuine signal, because cost concentration
and utilisation are properties of the claims process rather than of the demographic fields ruled
out earlier.

In [ ]:
# The basic shape of medical cost
df['Annual Medical Cost'].describe().round(0)

Median **2,103**, mean **3,033**, maximum **65,725**.

The mean sits 44% above the median, which happens one way: a small group costs far more than
everyone else, lifting the total without moving the member in the middle. That maximum is 31 times
a typical member.

This changes a real decision. Budgeting 3,033 per member over-provides for most of the book and
still gets caught by the extremes. For a typical member the answer is 2,103. **FIN**

In [ ]:
# Plotting cost on a log scale, since the tail makes a linear axis unreadable
plt.figure(figsize=(10, 6))
positive = df.loc[df['Annual Medical Cost'] > 0, 'Annual Medical Cost']
median_cost = df['Annual Medical Cost'].median()

plt.hist(positive, bins=np.logspace(1.5, 5, 70), color=BLUE)
plt.xscale('log')
plt.axvline(median_cost, color=ORANGE, lw=2)
plt.annotate(f'median {median_cost:,.0f}', (median_cost, plt.ylim()[1] * 0.9),
             xytext=(8, 0), textcoords='offset points', color=ORANGE, fontweight='bold')

plt.xlabel('Annual medical cost (log scale)')
plt.ylabel('Members')
plt.title('Medical cost is lognormal with a long right tail')
plt.grid(axis='y', alpha=0.5)
save('v2_09_cost_distribution')
plt.show()

##### Reading the distribution

The axis multiplies rather than adds, so 100-to-1,000 occupies the same width as 1,000-to-10,000.
Linear, the book would compress into the first inch.

One hump near 2,000, a thin thread running right, and the hump is roughly **symmetric** on this
axis — members are about as likely to cost half the typical amount as twice it.

The expensive members are therefore **not outliers to remove**; they continue the same pattern, so
they are ordinary members having an expensive year.

The shape also dictates model construction: fed raw, a model chases the largest numbers. Section 6
handles that, and the standard fix has a trap in it.

### How concentrated is the spending?

The tail matters more than its size suggests. Measuring exactly how much of the book's total spend
sits with how few members decides whether targeted intervention is worth doing at all.

In [ ]:
# What share of total spend each cost decile carries
df['Cost Decile'] = pd.qcut(df['Annual Medical Cost'], 10, labels=False) + 1
(df.groupby('Cost Decile')['Annual Medical Cost'].sum() / df['Annual Medical Cost'].sum() * 100).round(1)

In [ ]:
# The Lorenz curve, which shows the concentration as a whole
sorted_cost = np.sort(df['Annual Medical Cost'].values)
cumulative = np.cumsum(sorted_cost) / sorted_cost.sum()
share_of_members = np.arange(1, len(sorted_cost) + 1) / len(sorted_cost)
gini = 1 - 2 * np.trapezoid(cumulative, share_of_members)

plt.figure(figsize=(10, 6))
plt.plot(share_of_members * 100, cumulative * 100, color=BLUE, lw=2.5)
plt.plot([0, 100], [0, 100], color='grey', ls='--')
plt.fill_between(share_of_members * 100, cumulative * 100, share_of_members * 100, color=BLUE, alpha=0.12)

# Marking the points a business would actually care about
for pct in [50, 80, 90]:
    value = cumulative[int(len(sorted_cost) * pct / 100) - 1] * 100
    plt.scatter([pct], [value], color=ORANGE, zorder=5, s=50)
    plt.annotate(f'cheapest {pct}% of members\ncarry {value:.0f}% of spend', (pct, value),
                 xytext=(-8, 12), textcoords='offset points', ha='right', fontsize=10, color=INK_2)

plt.annotate(f'Gini {gini:.2f}', (58, 25), fontsize=14, color=BLUE, fontweight='bold')
plt.xlabel('Cumulative share of members, cheapest first (%)')
plt.ylabel('Cumulative share of medical spend (%)')
plt.title('A tenth of the members carry a third of the spend')
plt.grid(alpha=0.5)
save('v2_10_cost_concentration')
plt.show()

##### Reading the Lorenz curve

Members accumulate left to right, cheapest first. The dashed line is perfect evenness; the gap to
the blue curve is the finding.

The **top decile carries 33.5%** of everything paid out and the top two deciles **50.5%**. The
cheaper half of the membership accounts for about 12%.

**Gini 0.45** summarises that gap on a 0-to-1 scale — firmly in heavily-concentrated territory.

This is the most actionable structural fact here: a programme reaching 10% of members addresses a
third of the cost, which is what makes targeted intervention fundable. Evenly spread spend would
leave no efficient target and only across-the-board price rises. **CM / FIN**

Whether that decile is identifiable *in advance* is Model B.

### Is cost driven by claiming often, or claiming big?

Total cost is frequency times severity. Which of the two drives it changes what you would do about
it — high frequency suggests managing routine utilisation, high severity suggests catastrophic
cover and reinsurance.

In [ ]:
# How often members claim, and whether frequent claimers also claim larger amounts
claimants = df[df['Claims Count'] > 0]
severity = claimants.groupby('Claims Count').agg(members=('Id', 'size'),
                                                 median_claim=('Average Claim Amount', 'median'))
severity = severity[severity.members >= MIN_N]

plt.figure(figsize=(10, 6))
plt.plot(severity.index, severity.median_claim, marker='o', color=BLUE, lw=2.5)
plt.xlabel('Number of claims in the year')
plt.ylabel('Median size of each claim')
plt.title('Members who claim often claim small')
plt.grid(alpha=0.5)
save('v2_11_frequency_vs_severity')
plt.show()

##### Frequency or severity?

The line falls, so the two move in **opposite** directions. A member with one claim had a much
larger typical claim than one with eight.

That matches practice: one claim usually means one significant event, eight means routine
outpatient visits.

So cost here is driven by frequency rather than catastrophic single events, and the 38% who never
claim subsidise the rest. For care management that points at managing utilisation patterns rather
than chasing large claims. **CLM / CM**

### Does using more care actually cost more?

This sounds obvious, but I want to confirm it and see how steep each relationship is, because these
are the fields that would go into any predictive model — and unlike demographics, most of them are
observable before the year starts.

In [ ]:
# The utilisation measures to test, each with a sensible cap so thin tails do not distort the line
UTILISATION = [('Visits in Last Year', 'GP and outpatient visits', 12),
               ('Hospitalizations in Last 3 Years', 'Hospitalisations (3 years)', 3),
               ('Days Hospitalized in Last 3 Years', 'Days hospitalised (3 years)', 14),
               ('Medication Count', 'Medications', 10),
               ('Total Procedures', 'Total procedures', 14),
               ('Chronic Conditions Count', 'Chronic conditions', 6)]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))

for ax, (col, label, cap) in zip(axes.ravel(), UTILISATION):
    grouped = df.groupby(col).agg(cost=('Annual Medical Cost', 'median'), n=('Id', 'size'))
    grouped = grouped[(grouped.index <= cap) & (grouped.n >= MIN_N)]

    ax.plot(grouped.index, grouped.cost, marker='o', color=BLUE, lw=2.5)
    ax.fill_between(grouped.index, 0, grouped.cost, color=BLUE, alpha=0.1)
    ax.set_title(f'{label}   x{grouped.cost.iloc[-1] / grouped.cost.iloc[0]:.1f}', fontsize=12)
    ax.set_xlabel(label, fontsize=10)
    ax.set_ylabel('Median cost', fontsize=10)
    ax.set_ylim(0, grouped.cost.max() * 1.2)
    ax.grid(alpha=0.5)

fig.suptitle('Every utilisation measure rises steadily with cost', x=0.02, ha='left',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
save('v2_12_utilisation_dose_response')
plt.show()

##### Reading the six panels

Each panel groups members by one utilisation measure and plots the median cost of each group.

All six climb, and climb **without reversals** — every step up in usage is a step up in cost. The
consistency matters more than the steepness: a relationship holding at every level can be relied on
for an individual member, not just on average. Days hospitalised is steepest.

Kept together because the point is the consistency across all six; separate charts would lose it.

These are the variables worth modelling. They carry real signal, unlike the demographics — but only
the prior-history ones are knowable before the year starts. That distinction is not visible here
and it matters a great deal in section 6.

---

## 4.7 Which Condition Is Actually High Risk?

If care management has to pick a condition to focus on, which one? The instinctive answer is
whichever costs the most per patient. I want to test that instinct, because I think it produces
the wrong priority list.

In [ ]:
# What each condition costs a member, and what it costs the book overall
condition_rows = []
for cond in CONDITIONS:
    has, without = df[df[cond] == 1], df[df[cond] == 0]
    condition_rows.append({
        'condition': COND_SHORT.get(cond, cond),
        'members': len(has),
        'prevalence %': round(len(has) / len(df) * 100, 1),
        'extra cost per member': round(has['Annual Medical Cost'].median() - without['Annual Medical Cost'].median()),
        '% of all spend': round(has['Annual Medical Cost'].sum() / df['Annual Medical Cost'].sum() * 100, 1),
    })

conditions_cost = pd.DataFrame(condition_rows)
conditions_cost.sort_values('extra cost per member', ascending=False).reset_index(drop=True)

In [ ]:
# The same ten conditions, ranked by their share of total portfolio spend instead
conditions_cost.sort_values('% of all spend', ascending=False).reset_index(drop=True)

The ordering has almost completely inverted. That divergence is the finding, so it goes on one
chart.

In [ ]:
# Severity against burden, with each bubble sized by how common the condition is
plt.figure(figsize=(10, 6.5))
plt.scatter(conditions_cost['extra cost per member'], conditions_cost['% of all spend'],
            s=conditions_cost['prevalence %'] * 60, color=BLUE, alpha=0.55,
            edgecolor='white', linewidth=2, zorder=3)

# Nudging a couple of labels apart where the bubbles sit close together
NUDGE = {'Mental health': (-78, -4), 'Arthritis': (66, -6), 'Diabetes': (62, -4),
         'Asthma': (-66, -6), 'Cardiovascular': (78, -14), 'Cancer': (-34, 6), 'Kidney': (-16, 0)}

for _, r in conditions_cost.iterrows():
    dx, dy = NUDGE.get(r.condition, (0, 0))
    plt.annotate(f"{r.condition}\n{r['prevalence %']}% of members",
                 (r['extra cost per member'], r['% of all spend']),
                 xytext=(dx, np.sqrt(r['prevalence %'] * 60) / 2 + 9 + dy),
                 textcoords='offset points', ha='center', fontsize=10)

plt.xlabel('Extra cost per affected member')
plt.ylabel("Share of the book's total spend (%)")
plt.title('The condition worst for a member is not the one worst for the book')
plt.xlim(650, 1180)
plt.ylim(-2, 31)
plt.grid(alpha=0.5)
save('v2_13_severity_vs_burden')
plt.show()

##### Severity against burden

Horizontal is **severity**, extra cost per affected member. Vertical is **burden**, share of the
book's spend. Bubble size is prevalence.

**Severity barely varies** — 761 to 1,039 across all ten conditions, a spread of **1.37 times**.
**Burden varies 13.5 times**, from kidney disease at 2.0% of spend to hypertension at 26.8%.

Liver disease is worst to have at 1,039 extra, affects 1.5% of members, accounts for 2.1% of spend.
Hypertension costs slightly less per member and 20.4% of the book has it, so it takes 26.8% of
everything paid out.

**Prioritise by prevalence × cost, not severity.** A liver programme cannot move the needle,
whatever its clinical merit. **CM / ACT**

---

## 4.8 Which Age Group Actually Needs Insurance?

Average cost is a poor way to answer this. It rises gently with age and implies the young barely
need cover, which misunderstands what insurance is for. Insurance absorbs the **bad year** — so the
honest measure is the gap between a normal year and a bad one.

In [ ]:
# The typical year, the bad year and the catastrophic year, by age band
df['Age Band'] = pd.cut(df['Age'], [15, 29, 39, 49, 59, 69, 120],
                        labels=['16-29', '30-39', '40-49', '50-59', '60-69', '70+'])

exposure = df.groupby('Age Band', observed=True).agg(
    members=('Id', 'size'),
    income=('Income', 'median'),
    typical_year=('Annual Medical Cost', 'median'),
    bad_year=('Annual Medical Cost', lambda x: x.quantile(0.90)),
    catastrophic_year=('Annual Medical Cost', lambda x: x.quantile(0.99)))

# Stating the denominator explicitly: the catastrophic shortfall against the band's MEDIAN income
exposure['catastrophe gap'] = exposure.catastrophic_year - exposure.typical_year
exposure['catastrophe as % of median income'] = (exposure['catastrophe gap'] / exposure.income * 100).round(1)
exposure.round(0)

In [ ]:
# Charting the spread between a normal year and a catastrophic one, by age
plt.figure(figsize=(10, 6))
x = np.arange(len(exposure))

plt.fill_between(x, exposure.typical_year, exposure.catastrophic_year, color=RED, alpha=0.12,
                 label='Catastrophic year (99th percentile)')
plt.fill_between(x, exposure.typical_year, exposure.bad_year, color=ORANGE, alpha=0.22,
                 label='Bad year (90th percentile)')
plt.plot(x, exposure.typical_year, marker='o', color=BLUE, lw=2.5, label='Typical year (median)')
plt.plot(x, exposure.catastrophic_year, marker='o', color=RED, lw=2)
plt.plot(x, exposure.bad_year, marker='o', color=ORANGE, lw=2)

for i in [0, len(x) - 1]:
    plt.annotate(f'{exposure.catastrophic_year.iloc[i]:,.0f}', (i, exposure.catastrophic_year.iloc[i]),
                 xytext=(0, 8), textcoords='offset points', ha='center',
                 fontsize=11, color=RED, fontweight='bold')

plt.xticks(x, exposure.index)
plt.xlabel('Age band')
plt.ylabel('Annual medical cost')
plt.title('What a year can cost, by age')
plt.legend()
plt.grid(alpha=0.5)
save('v2_14_bad_year_by_age')
plt.show()

##### Interpretation

**Median** is a normal year, **p90** a bad one, **p99** catastrophic. The shaded gaps are what
insurance absorbs — a member can budget the normal year, not the gap above it.

The median rises only **1.61 times** across the whole age range, which is the weak argument most
people reach for first.

A 70-year-old faces nearly 20,000 against a typical 2,725 — **47% of the band's median income**.

**The unexpected result is at the young end.** A 16-29 member has a median cost of 1,688 and a
catastrophic year of 12,215 — **29% of annual median income**. Young, mostly healthy, still unable
to absorb a bad year.

Marketing to the young on expected cost loses, because their expected cost really is low.
Marketing on **volatility** is honest and far stronger. **PROD**

---

## 4.9 Does the Product Ladder Do Anything?

Members can buy Bronze, Silver, Gold or Platinum. Section 4.3 already told me that nothing about a
member predicts which one they hold, which is odd in itself. Now the other side of it: what does a
member get for climbing the ladder?

In [ ]:
# What each tier charges, against everything the member might be buying with it
ladder = df.groupby('Network Tier').agg(
    members=('Id', 'size'),
    median_premium=('Annual Premium', 'median'),
    provider_quality=('Provider Quality Rating', 'median'),
    tenure_years=('Policy Term (Years)', 'median'),
    policy_changes=('Policy Changes in Last 2 Years', 'mean'),
    visits=('Visits in Last Year', 'mean')).reindex(TIER)

# What share of their own medical spend each tier's members actually hand over
ladder['member pays %'] = (df.groupby('Network Tier')['Annual Premium'].sum()
                           / df.groupby('Network Tier')['Annual Medical Cost'].sum() * 100).reindex(TIER)
ladder['share of book %'] = (df['Network Tier'].value_counts(normalize=True) * 100).reindex(TIER)
ladder.round(2)

In [ ]:
# Indexing everything to Bronze, so it is obvious which lines actually move
indexed = pd.DataFrame({
    'What the member pays': ladder['member pays %'],
    'Provider quality received': ladder.provider_quality,
    'Tenure (loyalty)': ladder.tenure_years,
    'Care actually used': ladder.visits})
indexed = indexed / indexed.iloc[0] * 100

plt.figure(figsize=(10, 6))
for i, col in enumerate(indexed.columns):
    emphasis = i == 0
    plt.plot(range(4), indexed[col], marker='o',
             color=RED if emphasis else CAT[i], lw=3 if emphasis else 2,
             zorder=5 if emphasis else 3, label=col)
    plt.annotate(f'{indexed[col].iloc[-1]:.0f}', (3, indexed[col].iloc[-1]),
                 xytext=(9, 0), textcoords='offset points', va='center',
                 fontsize=11, fontweight='bold', color=RED if emphasis else CAT[i])

plt.axhline(100, color='grey', ls='--')
plt.xticks(range(4), TIER)
plt.xlabel('Network tier')
plt.ylabel('Indexed to Bronze = 100')
plt.title('Climbing the ladder costs more and delivers nothing measurable')
plt.legend(loc='upper left')
plt.grid(alpha=0.5)
save('v2_15_tier_ladder')
plt.show()

##### Reading the ladder

Indexed to Bronze at 100. **One line moves.** Platinum members hand over **24.4%** of their own
medical spend against **16.6%** on Bronze — 1.47 times the rate.

Provider quality is identical at 3.6 across all four tiers. Care consumed is flat. Median tenure on
Platinum is a year *shorter* than the other three.

On every observable dimension the ladder is a **cost-share dial wearing the costume of a benefit
ladder**.

The question for product: what does Platinum deliver that Bronze does not? If it is something
unrecorded — faster pre-authorisation, wider hospital list — we should start recording it. **Either
attach a measurable benefit to the upper tiers or collapse them. PROD**

### Does where a member lives change what they use?

The last product-adjacent question is whether geography changes consumption, which would justify a
network strategy.

In [ ]:
# The full utilisation basket across settlement types, as a deviation from the book average
UTIL_COLS = ['Visits in Last Year', 'Hospitalizations in Last 3 Years',
             'Days Hospitalized in Last 3 Years', 'Medication Count',
             'Imaging Procedures Count', 'Surgical Procedures Count',
             'Lab Procedures Count', 'Consultation Procedures Count', 'Claims Count']

settlement = df.groupby('Urban / Rural')[UTIL_COLS].mean().reindex(['Urban', 'Suburban', 'Rural'])
deviation = (settlement / df[UTIL_COLS].mean() - 1) * 100

plt.figure(figsize=(10, 6))
plt.axhspan(-10, 10, color=BLUE, alpha=0.08)
for i, place in enumerate(deviation.index):
    plt.plot(range(len(UTIL_COLS)), deviation.loc[place], marker='o', color=CAT[i], label=place)

plt.axhline(0, color=INK_2)
labels = [c.replace(' in Last Year', '').replace(' in Last 3 Years', ' (3y)')
           .replace(' Procedures Count', '').replace(' Count', '') for c in UTIL_COLS]
plt.xticks(range(len(UTIL_COLS)), labels, rotation=40, ha='right')
plt.ylabel('Deviation from the book average (%)')
plt.ylim(-14, 14)
plt.title('Urban, suburban and rural members consume almost identical care')
plt.legend()
plt.grid(axis='y', alpha=0.5)
save('v2_16_consumption_by_settlement')
plt.show()

Every measure sits inside the ±10% band. Those with enough volume to be reliable are flat to well
under 1%: visits differ by **0.86%** across settlement types, claims by **0.68%**, medication by
**0.14%**.

The two straying furthest — days hospitalised and surgical procedures, around 8% and 6% — both
average **under 0.4 per member**. On a base that small, a few hundredths of an event looks dramatic
as a percentage and is trivial in absolute terms.

**Where a member lives does not change the care they consume.** With the flat regional costs from
4.4, network and regional strategy cannot be built from this data; that needs provider-level
identifiers and finer geography than five regions. **NET**

---

## 4.10 Who Should We Care-Manage First?

Two things from earlier combine here. Cost is concentrated in a small group, and burden follows
prevalence rather than severity. Together they should tell me which cohorts are worth enrolling.

For each candidate group I want three numbers: how many members it reaches, how much of the book's
spend sits inside it, and how likely those members are to end up in the expensive decile.

In [ ]:
# The cohorts a care-management team could realistically target
threshold = df['Annual Medical Cost'].quantile(0.90)

COHORTS = {
    'Hypertensive': df['Hypertension'] == 1,
    '2+ chronic conditions': df['Chronic Conditions Count'] >= 2,
    'Had a major procedure': df['Had Major Procedure'] == 1,
    '4+ visits last year': df['Visits in Last Year'] >= 4,
    'Mental health condition': df['Mental Health Condition'] == 1,
    'Current smoker': df['Smoker Status'] == 'Current',
    'Hospitalised in last 3 years': df['Hospitalizations in Last 3 Years'] > 0,
    'Aged 70+': df['Age'] >= 70,
    'Undiagnosed diabetic': undiagnosed,
}

targeting = pd.DataFrame([{
    'cohort': name,
    'members': int(mask.sum()),
    '% of book': round(mask.mean() * 100, 1),
    '% of spend': round(df.loc[mask, 'Annual Medical Cost'].sum() / df['Annual Medical Cost'].sum() * 100, 1),
    '% in top decile': round((df.loc[mask, 'Annual Medical Cost'] > threshold).mean() * 100, 1),
} for name, mask in COHORTS.items()]).sort_values('% of spend', ascending=False).reset_index(drop=True)
targeting

In [ ]:
# Reach against return, so the trade-off is visible
plt.figure(figsize=(10, 6.5))
plt.scatter(targeting['% of book'], targeting['% of spend'],
            s=targeting['% in top decile'] * 24, color=BLUE, alpha=0.55,
            edgecolor='white', linewidth=2, zorder=3)

limit = max(targeting['% of book'].max(), targeting['% of spend'].max()) * 1.15
plt.plot([0, limit], [0, limit], color='grey', ls='--')
plt.annotate('no concentration\n(spend share = member share)', (limit * 0.7, limit * 0.7),
             rotation=33, fontsize=10, color=INK_2, ha='center')

for _, r in targeting.iterrows():
    plt.annotate(r.cohort, (r['% of book'], r['% of spend']),
                 xytext=(0, np.sqrt(r['% in top decile'] * 24) / 2 + 8),
                 textcoords='offset points', ha='center', fontsize=10)

plt.xlabel('Reach — share of the membership in this cohort (%)')
plt.ylabel('Return — share of total medical spend it accounts for (%)')
plt.title('Cohorts above the line concentrate more spend than their headcount')
plt.xlim(0, limit)
plt.ylim(0, limit)
plt.grid(alpha=0.5)
save('v2_17_care_management_targeting')
plt.show()

##### Reach against return

Horizontal is **reach**, vertical is **return**, the dashed line is parity, bubble size is the
chance of landing in the top cost decile.

Every cohort sits above the line, which only says sick members cost more. What decides a budget is
how far above, and where each sits on reach.

**Hypertension is the volume play** — 20.4% of members, 26.8% of spend, the largest pool available.

**Prior hospitalisation is the precision play** — 9% of members, but **25% land in the most
expensive tenth**, two and a half times the base rate. Enrol a hundred and about twenty-five turn
out genuinely expensive, against ten at random.

**Build on prior hospitalisation and multi-morbidity; run hypertension separately for reach. CM**

---

## 4.11 Which Package Suits Which Member?

Bringing 4.8 and 4.10 together. Only age and clinical burden legitimately segment this book, so
those are the two axes. And 4.8 established that the right measure is downside exposure, not
average cost.

In [ ]:
# Segments built from the only two variables that legitimately stratify this book
segment_frame = df[df['Age_Life_Stage'] != '0-15 (Minors)'].copy()
segment_frame['Burden'] = np.where(segment_frame['Chronic Conditions Count'] >= 2, '2+ conditions',
                          np.where(segment_frame['Chronic Conditions Count'] == 1, '1 condition', 'No condition'))

segments = segment_frame.groupby(['Age_Life_Stage', 'Burden'], observed=True).agg(
    members=('Id', 'size'),
    typical_year=('Annual Medical Cost', 'median'),
    bad_year=('Annual Medical Cost', lambda x: x.quantile(0.90)))
segments = segments[segments.members >= MIN_N]
segments['exposure gap'] = segments.bad_year - segments.typical_year
segments.round(0)

In [ ]:
# The gap between a normal year and a bad one, for every segment
seg = segments.reset_index()
seg['stage_order'] = seg['Age_Life_Stage'].map({s: i for i, s in enumerate(LIFE_STAGE)})
seg = seg.dropna(subset=['stage_order']).sort_values(['stage_order', 'Burden'])
seg['label'] = seg['Age_Life_Stage'].str.split(' \(').str[0] + '  ·  ' + seg['Burden']

plt.figure(figsize=(10, 7))
y = np.arange(len(seg))
plt.hlines(y, seg.typical_year, seg.bad_year, color=GRID_C, lw=7)
plt.scatter(seg.typical_year, y, color=BLUE, s=90, zorder=5, label='Typical year')
plt.scatter(seg.bad_year, y, color=RED, s=90, zorder=5, label='Bad year (90th percentile)')

for i, (lo, hi) in enumerate(zip(seg.typical_year, seg.bad_year)):
    plt.annotate(f'{lo:,.0f}', (lo, i), xytext=(-8, 0), textcoords='offset points',
                 ha='right', va='center', fontsize=10, color=BLUE, fontweight='bold')
    plt.annotate(f'{hi:,.0f}', (hi, i), xytext=(8, 0), textcoords='offset points',
                 ha='left', va='center', fontsize=10, color=RED, fontweight='bold')

plt.yticks(y, seg.label)
plt.xlabel('Annual medical cost')
plt.title('The gap between a normal year and a bad one is what cover has to absorb')
plt.xlim(0, seg.bad_year.max() * 1.2)
plt.legend(loc='lower right')
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.5)
save('v2_18_package_fit_by_segment')
plt.show()

##### What I would recommend from this

The bars widen down the chart, and the width is what the member cannot absorb.

A healthy under-30 has a typical year of 1,282 and a bad year of 3,627 — a gap near 2,300. A 65-plus
member with two or more conditions runs 3,828 to 10,894, a gap over 7,000.

- **Under 50, no condition:** gap small enough to part-self-insure. **High-deductible Bronze** is
  rational, since the formula charges only 1% of the deductible — raising it is nearly free.
- **Any age, one condition:** around 6,500 in a bad year. **Silver**.
- **50-plus, two or more conditions:** the widest gap in the book. **Low-deductible Gold or
  Platinum** earns its cost here.
- **Any prior hospitalisation:** three times more likely to land in the expensive decile whatever
  the plan. A care-management case, not a product one.

One caveat throughout: because premium is a fixed share of realised cost, **no tier is objectively
better value than another** here. This matches cover to exposure, which is real. It says nothing
about pricing efficiency, which this data cannot. **PROD / ACT**

---

## 4.12 Can We Trust Our Own Fields?

Before any of this feeds a model, I want to check the fields that look most useful, because the
ones correlating most strongly with cost are exactly the ones most likely to be circular.

`Risk Score` is the obvious candidate. If it is calculated from member attributes, it is a
legitimate underwriting input. If it is calculated from cost or claims, it is leakage.

In [ ]:
# Testing whether Risk Score can be rebuilt from clinical attributes alone
OBSERVABLE = ['Age', 'Chronic Conditions Count', 'Systolic Blood Pressure', 'Diastolic Blood Pressure',
              'BMI', 'HbA1c Level', 'LDL Cholesterol', 'Visits in Last Year',
              'Hospitalizations in Last 3 Years', 'Medication Count'] + CONDITIONS

X_tr, X_te, y_tr, y_te = train_test_split(df[OBSERVABLE], df['Risk Score'], test_size=0.3, random_state=42)
round(r2_score(y_te, LinearRegression().fit(X_tr, y_tr).predict(X_te)), 3)

**0.86.** Age, chronic count, blood pressure and the other clinical fields rebuild 86% of `Risk
Score`.

So it is built from things observed before cost is incurred — a legitimate underwriting input, not
a field containing the answer.

It is also, on that evidence, nearly redundant: 86% of it already sits in fields a model would
have.

In [ ]:
# How much each group of fields adds to a cost model, which is how leakage shows itself
def cost_model_r2(features):
    X_tr, X_te, y_tr, y_te = train_test_split(df[features], log_cost, test_size=0.3, random_state=42)
    return round(r2_score(y_te, LinearRegression().fit(X_tr, y_tr).predict(X_te)), 3)


pd.Series({
    'Observable clinical fields only': cost_model_r2(OBSERVABLE),
    'Plus Risk Score': cost_model_r2(OBSERVABLE + ['Risk Score']),
    'Plus the claims fields': cost_model_r2(OBSERVABLE + ['Risk Score', 'Total Claims Paid', 'Average Claim Amount']),
})

##### What the three models say

Clinical fields alone: **0.164** — matching the 16% ceiling from 4.2 by a different route.

Plus `Risk Score`: **0.190**. Small, as expected from a field 86% rebuildable from inputs already
present.

Plus the claims fields: **0.435**. No new information about any member arrived between those rows —
those fields are computed from the cost being predicted.

**Leakage is dangerous because it does not look like a bug, it looks like a better model.**

Rule for section 6: **exclude** premium, monthly premium, total claims paid, average claim amount
and loss ratio. **`Risk Score` may stay for reporting**, but models get its components instead.

---

## 4.13 What I Found

### Supported

1. **Pricing is arithmetic, not underwriting.** `200 + 0.01 × Deductible + tier_rate × Cost`
   reproduces 99.997% of premiums to the cent. Invalidates any pricing-adequacy analysis. **ACT / FIN**
2. **Cost is concentrated and clinically driven.** Top decile carries 33.5% of spend; membership
   goes with hospitalisation, multi-morbidity, procedures and age. **CM**
3. **Burden follows prevalence, not severity.** 1.37× spread in cost per member against 13.5× in
   share of spend. **CM / ACT**
4. **Insurance value is volatility.** Median cost rises 1.61× with age, but a 16-29 member faces a
   catastrophic year worth 29% of income. **PROD**
5. **Only age and clinical burden segment this book.** 64 of 1,326 pairs are genuine, all clinical,
   utilisation or family composition. **UW**

### Not supported by this data

| Question | Why not |
|:---|:---|
| Is the book priced adequately? | Premium is computed from realised cost |
| Which regions carry more risk? | Regional cost, morbidity and utilisation are flat |
| Why do unemployed people buy cover? | That cohort matches the book average on every field |
| Does anything drive plan choice? | Demographic-to-product association is about 0.01 |
| Do lifestyle factors drive morbidity? | Smokers and never-smokers have near-identical rates |
| Does multi-morbidity compound? | Conditions co-occur at a lift of 1.00 |
| Anything paediatric | Cleaning left 35 members under 18 |

### Recommended

1. **Underwrite on age, chronic count and prior utilisation** — the only variables that move cost.
2. **Stop segmenting on region, employment, education, household size, settlement.** Each under 1%.
3. **Build care management on prior hospitalisation and multi-morbidity**, hypertension separately.
4. **Escalate the 273 undiagnosed diabetics** — the lab value is already on file.
5. **Ask product what the upper tiers deliver.** 1.47× the contribution rate, nothing measurable.
6. **Flag the regressive contribution structure.** Poorest quintile pays 8.8× the income share.
7. **Drop premium and claims fields before modelling** — they lift R² from 0.16 to 0.44 by leaking.

### Worth confirming with whoever produced the extract

The ten conditions occur independently (lift 1.00 between every pair). Smoking drives cost but
shows no relationship to any recorded condition. Utilisation is identical across every geography.
Cancer history is as common at 25 as at 75. Premium is closed-form on realised cost.

Each is what the data says. None changes the method; several would change what the findings mean.

---

# 5. The Dashboard Picture

Section 4 is long, because working out what a book contains takes a lot of small checks. This
section is the opposite: the ten charts I would actually put in front of a room, drawn once,
interactive, in one place.

They are Plotly rather than matplotlib, which is a deliberate choice rather than a second style
for its own sake. Hovering to read an exact value matters far more in a meeting than it does
inside a written analysis, and these are the charts people will want to interrogate.

Everything here recomputes from the same frame the analysis used, and most of it reuses the tables
section 4 already built — `conditions_cost`, `targeting`, `exposure`, `effect`, `burden`. That is
the payoff for having one notebook rather than six: the dashboard cannot drift away from the
analysis, because there is only one set of numbers.

Nothing in this section is written to disk. These charts live in the notebook.

In [ ]:
# Plotly, for charts a reader can hover over
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = 'plotly_mimetype+notebook_connected'

In [ ]:
# A house style for this section, matching the colour-blind palette used above
PLOTLY_PALETTE = ['#0072B2', '#E69F00', '#009E73', '#D55E00', '#CC79A7', '#56B4E9', '#F0E442']


def style(fig, title, subtitle, height=480):
    fig.update_layout(
        template='plotly_white',
        title={'text': f'{title}<br><sup>{subtitle}</sup>', 'x': 0.02, 'xanchor': 'left'},
        height=height,
        margin={'l': 60, 'r': 30, 't': 80, 'b': 50},
        colorway=PLOTLY_PALETTE,
    )
    return fig

## 5.1 Cost distribution

The shape everything else follows from: right-skewed, long tailed, most members clustered around
2,100.

In [ ]:
cost = df['Annual Medical Cost']
p50, p90 = cost.median(), cost.quantile(0.9)

# Pre-binning on a log grid, so 96,639 points become 60 readable bars
log_bins = np.logspace(np.log10(cost.min()), np.log10(cost.max()), 61)
counts, edges = np.histogram(cost, bins=log_bins)
centres = np.sqrt(edges[:-1] * edges[1:])          # geometric midpoints sit correctly on a log axis

fig = go.Figure(go.Bar(x=centres, y=counts, width=np.diff(edges),
                       marker_color=PLOTLY_PALETTE[0], name='Members'))
fig.update_xaxes(type='log', title='Annual medical cost (log scale)')
fig.update_yaxes(title='Members')
fig.add_vline(x=p50, line_dash='dash', line_color='#333333',
              annotation_text=f'median {p50:,.0f}', annotation_position='top left')
fig.add_vline(x=p90, line_dash='dash', line_color=PLOTLY_PALETTE[3],
              annotation_text=f'p90 {p90:,.0f}', annotation_position='top right')
style(fig, 'Cost Distribution',
      'Annual medical cost, log scale — the top decile carries 33.5% of total spend')
fig.show()

## 5.2 Spend concentration

The same fact stated as a Lorenz curve, which is the version that persuades a budget meeting: a
tenth of the members carry a third of the money.

In [ ]:
sorted_costs = np.sort(df['Annual Medical Cost'].to_numpy(float))
cum_spend = np.concatenate([[0.0], np.cumsum(sorted_costs) / sorted_costs.sum()])
cum_pop = np.linspace(0, 1, len(cum_spend))

fig = go.Figure()
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', name='Perfect equality',
                         line=dict(dash='dash', color='#999999')))
# Decimating to a 1,001-point grid keeps the figure light without changing the curve
idx = np.unique(np.linspace(0, len(cum_pop) - 1, 1001).astype(int))
fig.add_trace(go.Scatter(x=cum_pop[idx], y=cum_spend[idx], mode='lines', name='Observed',
                         line=dict(color=PLOTLY_PALETTE[0], width=3), fill='tonexty',
                         fillcolor='rgba(0,114,178,0.12)'))
fig.add_annotation(x=0.35, y=0.75, text=f'Gini = {gini:.2f}', showarrow=False,
                   font=dict(size=18, color=PLOTLY_PALETTE[3]))
fig.update_xaxes(title='Cumulative share of members')
fig.update_yaxes(title='Cumulative share of spend')
style(fig, 'Spend Concentration', 'Lorenz curve of annual medical cost')
fig.show()

## 5.3 What actually drives cost

The variance league from 4.2, with the 1% line that separates the four usable variables from the
ten that are not.

In [ ]:
league_sorted = effect['eta squared %'].sort_values()

fig = go.Figure(go.Bar(
    x=league_sorted.values, y=league_sorted.index, orientation='h',
    marker_color=[PLOTLY_PALETTE[3] if v >= 1 else PLOTLY_PALETTE[0] for v in league_sorted.values],
    text=[f'{v:.2f}%' for v in league_sorted.values], textposition='outside',
))
fig.update_xaxes(title='Share of cost variance explained (%)', range=[0, 11])
style(fig, 'What Actually Drives Cost',
      'Clinical burden, smoking, procedures and age clear 1%. Every demographic is inert', height=560)
fig.show()

## 5.4 Loss ratio by tier

Included with its caveat attached to the chart itself, because this is the chart somebody will
screenshot. The tier medians are simply `1 / tier_rate` from the formula in 4.1 — **arithmetic, not
evidence of underpricing.**

In [ ]:
loss_ratio = df.groupby('Network Tier')['Loss Ratio'].median().reindex(TIER)

fig = go.Figure(go.Bar(
    x=loss_ratio.index, y=loss_ratio.values,
    marker_color=[PLOTLY_PALETTE[1], '#9E9E9E', '#D4AF37', '#7B7FD4'],
    text=[f'{v:.2f}x' for v in loss_ratio.values], textposition='outside',
))
fig.update_yaxes(title='Median loss ratio (cost / premium)')
style(fig, 'Loss Ratio by Tier',
      'CAVEAT: premium = 200 + 0.01xDeductible + tier_rate x Cost, so each bar is 1/tier_rate — arithmetic, not inadequacy')
fig.show()

## 5.5 Age and morbidity

The one demographic gradient that survives. Five conditions shown rather than all ten, because the
other five are flat and add nothing but ink.

In [ ]:
GRADIENT_CONDITIONS = ['Hypertension', 'Diabetes', 'Arthritis',
                       'Mental Health Condition', 'Cardiovascular Disease']

fig = go.Figure()
for i, cond in enumerate(GRADIENT_CONDITIONS):
    fig.add_trace(go.Scatter(x=[c.split(' (')[0] for c in AGE_COHORT], y=by_age[cond],
                             mode='lines+markers', name=cond,
                             line=dict(color=PLOTLY_PALETTE[i], width=2.5)))
fig.update_yaxes(title='Prevalence (%)')
fig.update_layout(legend_title_text='Condition')
style(fig, 'Age-Morbidity Gradient',
      'Hypertension climbs 1.56x from 16-29 to 70+ — a real shift, but most older members still do not have it')
fig.show()

## 5.6 Severity against burden

The inversion that decides care-management priorities: the worst condition to *have* is not the
worst condition for the *book*.

In [ ]:
fig = go.Figure(go.Scatter(
    x=conditions_cost['extra cost per member'], y=conditions_cost['% of all spend'],
    mode='markers+text', text=conditions_cost['condition'], textposition='top center',
    marker=dict(size=conditions_cost['prevalence %'] * 2.2 + 8, color=PLOTLY_PALETTE[0], opacity=0.65),
    hovertemplate='%{text}<br>+%{x:,.0f} per member<br>%{y:.1f}% of spend<extra></extra>',
))
fig.update_xaxes(title='Extra cost per member with the condition (median uplift)')
fig.update_yaxes(title='Share of total portfolio spend (%)')
style(fig, 'Severity vs Burden',
      'Bubble size is prevalence. Liver disease is worst per member; hypertension dominates the book by volume')
fig.show()

## 5.7 Care-management targeting

Which cohorts are worth enrolling, ranked by the share of spend they reach.

In [ ]:
cohort_chart = targeting.sort_values('% of spend')

fig = go.Figure(go.Bar(
    x=cohort_chart['% of spend'], y=cohort_chart['cohort'], orientation='h',
    marker_color=PLOTLY_PALETTE[0],
    text=[f'{s:.1f}% of spend · {n:,} members'
          for s, n in zip(cohort_chart['% of spend'], cohort_chart['members'])],
    textposition='outside',
))
fig.update_xaxes(title='Share of total portfolio spend (%)',
                 range=[0, cohort_chart['% of spend'].max() * 1.4])
style(fig, 'Care-Management Targeting',
      'Prior hospitalisation reaches 9% of the book; the 273 undiagnosed diabetics are a governance item, not a financial one',
      height=520)
fig.show()

## 5.8 Affordability burden

Median income against the share of it that goes on premium. The bars rise 7.8 times; the line does
the opposite.

In [ ]:
income_median = df.groupby('Income Band', observed=True)['Income'].median()

fig = go.Figure()
fig.add_trace(go.Bar(x=income_median.index.astype(str), y=income_median.values,
                     name='Median income', marker_color=PLOTLY_PALETTE[0],
                     text=[f'{v:,.0f}' for v in income_median.values], textposition='outside'))
fig.add_trace(go.Scatter(x=burden.index.astype(str), y=burden.values,
                         name='Premium burden (% of income)', yaxis='y2',
                         mode='lines+markers', line=dict(color=PLOTLY_PALETTE[3], width=3),
                         marker=dict(size=10)))
fig.update_layout(
    yaxis=dict(title='Median income'),
    yaxis2=dict(title='Premium burden (% of income)', overlaying='y', side='right',
                range=[0, burden.max() * 1.4]),
    legend=dict(x=0.02, y=0.98),
)
style(fig, 'Affordability Burden',
      'Median premium is flat near 465 while income rises 7.8x — the poorest quintile carries 8.8x more of its income')
fig.show()

## 5.9 Regional exposure

Where the book sits, with mean cost printed on each bar so nobody reads a risk story into a
distribution fact.

In [ ]:
regions = (df.groupby('Region')
             .agg(members=('Id', 'count'), mean_cost=('Annual Medical Cost', 'mean'))
             .assign(share=lambda d: d['members'] / len(df) * 100)
             .sort_values('share', ascending=False))

fig = go.Figure(go.Bar(
    x=regions.index, y=regions['share'], marker_color=PLOTLY_PALETTE[0],
    text=[f'{s:.1f}%<br><sup>mean cost {c:,.0f}</sup>' for s, c in zip(regions['share'], regions['mean_cost'])],
    textposition='outside',
))
fig.update_yaxes(title='Share of members (%)', range=[0, regions['share'].max() * 1.3])
style(fig, 'Regional Exposure',
      'South holds 28% of the book against Central 12%, but mean cost is flat to about 2% — where to sell, not what to charge')
fig.show()

## 5.10 What a bad year costs, by age

The volatility story from 4.8, and the chart behind the bad-year model in section 6.

In [ ]:
fig = go.Figure()
for series, name, colour, dash in [
        (exposure['typical_year'], 'Typical year (p50)', PLOTLY_PALETTE[0], 'solid'),
        (exposure['bad_year'], 'Bad year (p90)', PLOTLY_PALETTE[1], 'dash'),
        (exposure['catastrophic_year'], 'Catastrophic year (p99)', PLOTLY_PALETTE[3], 'dot')]:
    fig.add_trace(go.Scatter(x=exposure.index.astype(str), y=series, mode='lines+markers',
                             name=name, line=dict(color=colour, width=2.5, dash=dash)))
fig.update_yaxes(title='Annual medical cost')
fig.update_xaxes(title='Age band')
style(fig, 'Bad-Year Cost by Age',
      'A bad year costs roughly 3x a typical one at every age — the gap is what cover exists to absorb')
fig.show()

That is the whole story in ten charts: a concentrated, clinically driven book with a pricing
formula rather than a price, a regressive contribution structure, a product ladder that delivers
nothing measurable, and one clear care-management target.

Everything above describes what already happened. The remaining question is whether any of it can
be known **in advance**, which is what section 6 is for.

---

# 6. Predicting Cost and Risk

Two facts from section 4 make modelling worth doing: cost is concentrated in a tenth of members,
and membership of that tenth goes with things observable in advance.

**Model A** predicts what a member will cost — the basis for a technical premium.
**Model B** ranks members by their chance of landing in the expensive tenth — a care-management
enrolment list.

**No premium model.** 4.1 recovered `Annual Premium` exactly, so training on it would reproduce
arithmetic already written down. The useful form of "what should we charge" is "what will this
member cost". Demonstrated in 6.5 rather than asserted.

## The Analysis Frame Is Not the Modelling Frame

`df` **describes what already happened**, so it may look at realised cost and premium. A model
cannot. Three different problems come out:

**1. Leaked fields.** Premium is computed from cost; claims paid is the insurer's share of it. A
model given these inverts arithmetic, then fails on a member whose cost is unknown.

**2. Same-period fields.** `Claims Count`, `Visits in Last Year` and the procedure counts are
recorded *in the same year as the cost being predicted* — not leakage in the strict sense, but
unknowable at renewal. Priced in 6.1.

**3. Re-encodings.** `Age Cohort` is `Age` binned; `Cost Decile` is the target in disguise.

What remains is `df_ml`: **renewal-legal**, with engineered features on top. Section 4 decided which
were worth engineering; no column crossed over untested.

## Setting Up

Everything the modelling needs that is not already bound. The estimators, the pipeline machinery
and the calibration tools arrive here; the metrics and split functions came in with section 4, so
they are not imported again. The engineered
feature block lives in `scripts/model_features.py` rather than in this notebook, and the reason is
specific: joblib pickles a notebook-defined function by reference to `__main__`, so the saved model
would fail to load in a fresh process. Importing it from a module keeps the round-trip test at the
end honest.

In [ ]:
# Adding what the modelling needs, plus the engineered feature block from scripts/
import json
import sys
from datetime import datetime, timezone

import joblib
import shap
import sklearn
import statsmodels.api as sm
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

sys.path.insert(0, str(ROOT / 'scripts'))
from model_features import ENGINEERED_NAMES, add_engineered

In [ ]:
# Settings in one place, so they are easy to change and easy to find
RANDOM_STATE = 42
TEST_SIZE = 0.25
TAIL_Q = 0.90          # "high cost" means above this point in the cost distribution
CAPACITY_PCT = 0.10    # what share of the book care management can actually enrol
N_JOBS = -1
TARGET = 'Annual Medical Cost'

## Writing Down the Feature Contract

Three lists, each with the reason attached to every entry. I would rather spell out why a field is
barred than leave a future reader guessing, because the failure this prevents is invisible: a model
trained with leakage does not crash or warn, it just reports a wonderful score.

In [ ]:
# Fields calculated FROM the cost I am predicting, with the reason each one is barred
LEAKY = {
    'Annual Premium': '= 200 + 0.01 x Deductible + tier_rate x Annual Medical Cost',
    'Monthly Premium': '= Annual Premium / 12, so equally derived',
    'Total Claims Paid': "the insurer's share OF the target, bounded by it",
    'Average Claim Amount': 'Total Claims Paid divided by Claims Count',
    'Loss Ratio': '= cost / premium, built during the exploratory work',
    'Risk Score': '86% rebuildable from clinical inputs, so it adds little over them',
    'Is High Risk': 'a thresholded version of Risk Score',
}
pd.Series(LEAKY, name='why it is barred').to_frame()

In [ ]:
# Fields recorded in the SAME year as the cost, so unknowable at the moment of prediction
SAME_PERIOD = {
    'Claims Count': 'the frequency leg of total_claims_paid = count x average',
    'Visits in Last Year': 'this year\'s visits, counted after the year happened',
    'Imaging Procedures Count': 'this year\'s procedures',
    'Surgical Procedures Count': 'this year\'s procedures',
    'Physiotherapy Procedures Count': 'this year\'s procedures',
    'Consultation Procedures Count': 'this year\'s procedures',
    'Lab Procedures Count': 'this year\'s procedures',
    'Had Major Procedure': 'derived from this year\'s surgical count',
}
pd.Series(SAME_PERIOD, name='why it cannot be used at renewal').to_frame()

In [ ]:
# Columns that exist only to serve the analysis above: identifiers and re-encodings
ANALYSIS_ONLY = ['Id', 'Plan Type (Full Name)', 'Age Groups', 'Age Cohort (10y)', 'Age_Life_Stage',
                 'Age Band', 'Glycemic Status', 'Diagnosis Status', 'Diabetes Clinical Quadrant',
                 'Alcohol_Missing', 'Income Band', 'Cost Decile', 'Top Decile', 'Total Procedures',
                 'Burden %']

RENEWAL_COLS = [c for c in df.columns
                if c not in set(LEAKY) | set(SAME_PERIOD) | set(ANALYSIS_ONLY) | {TARGET}]
len(RENEWAL_COLS)

## Building `df_ml`

The modelling frame, and the two targets that come off it. `y_cost` is what a member cost;
`y_high` is whether they landed in the most expensive tenth.

In [ ]:
# The modelling frame: raw renewal-legal columns only, engineering happens inside the pipeline
df_ml = df[RENEWAL_COLS].copy()
y_cost = df[TARGET].astype('float64')
y_high = (y_cost > y_cost.quantile(TAIL_Q)).astype(int)

df_ml.shape

The engineered block runs **inside** the pipeline, not on the frame. Engineered here, the saved
model would only accept data already engineered the same way, and the next user would have to
reconstruct that from memory. Inside, it takes raw member records — proven by the round-trip test
in 6.6.

Two properties are deliberate. Every feature is **row-wise**, computed from one member's own
fields with nothing pooled across members, so it cannot carry test-set information into training
whichever side of the split it runs. And every input is renewal-legal, which the guard below
re-checks rather than trusts.

In [ ]:
# Working out the column types after engineering, since the pipeline needs to know them
engineered_preview = add_engineered(df_ml)
NUM_COLS = engineered_preview.select_dtypes(include=[np.number]).columns.tolist()
CAT_COLS = [c for c in engineered_preview.columns if c not in NUM_COLS]

pd.Series({'raw renewal-legal columns': len(RENEWAL_COLS),
           'engineered columns added': len(ENGINEERED_NAMES),
           'numeric features': len(NUM_COLS),
           'categorical features': len(CAT_COLS)})

In [ ]:
# The guard. If a barred field ever creeps back in, this stops the notebook rather than
# letting it report a flattering score.
barred = set(LEAKY) | set(SAME_PERIOD) | {TARGET}
leaked = sorted(set(engineered_preview.columns) & barred)
assert not leaked, f'LEAKAGE: {leaked} must not reach a model'
assert 'Id' not in engineered_preview.columns, 'the member identifier is not a feature'

f'{len(NUM_COLS) + len(CAT_COLS)} features cleared the contract'

I wrote that as an `assert` rather than a comment because of what it prevents. If somebody later
adds a field back without thinking, I would rather the notebook stopped than quietly produced a
number nobody could reproduce.

### What leakage actually does to a score

Rather than assert that leakage matters, let me measure it. Same model, same data, same settings.
The only difference is whether the barred fields are included.

In [ ]:
# Fitting the same model with and without the barred fields, purely to compare
log_target_cost = np.log1p(y_cost)


def quick_r2(frame):
    encoded = pd.get_dummies(frame, drop_first=True)
    X_tr, X_te, y_tr, y_te = train_test_split(encoded, log_target_cost,
                                              test_size=TEST_SIZE, random_state=RANDOM_STATE)
    model = HistGradientBoostingRegressor(max_iter=300, early_stopping=True,
                                          random_state=RANDOM_STATE).fit(X_tr, y_tr)
    return r2_score(y_te, model.predict(X_te))


with_leakage = df[[c for c in df.columns if c not in ANALYSIS_ONLY + [TARGET]]]

pd.Series({'With the premium and claims fields': round(quick_r2(with_leakage), 4),
           'Without them, renewal-legal only': round(quick_r2(df_ml), 4)})

##### Interpretation

**With the barred fields: 0.999.** Individual medical cost is driven substantially by chance, so no
set of member attributes can account for 99.9% of it. That number should trigger suspicion, not
congratulation.

**Renewal-legal only: 0.21.** The honest figure, slightly above the 16% ceiling from 4.2 — expected,
since this has the engineered features and the full legal set behind it where the ceiling test used
five raw drivers.

No information about any member arrived between those runs. The first model was handed
`Annual Premium` and inverted the formula.

**Leakage does not look like a mistake, it looks like success.** A model reporting 0.999 passes a
review that 0.21 would struggle with, and only one of them works on a member whose cost is unknown.

## Preparing the Data

The categorical fields hold text and the models need numbers. The standard conversion turns each
category into its own yes/no column: `Region` becomes five columns with a 1 in the one that
applies.

That conversion goes inside the pipeline too, alongside the engineering, for the same reason.

In [ ]:
# The preparation steps that travel inside every model
def make_preprocessor(columns=None, scale=False):
    nums = [c for c in NUM_COLS if columns is None or c in columns or c in ENGINEERED_NAMES]
    cats = [c for c in CAT_COLS if columns is None or c in columns]

    numeric_steps = [('impute', SimpleImputer(strategy='median'))]
    if scale:                       # straight-line models need this, tree models do not
        numeric_steps.append(('scale', StandardScaler()))

    return ColumnTransformer([
        ('num', Pipeline(numeric_steps), nums),
        ('cat', Pipeline([
            ('impute', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='infrequent_if_exist',
                                     sparse_output=False, min_frequency=0.01)),
        ]), cats),
    ], remainder='drop', verbose_feature_names_out=False)

In [ ]:
# The full front end: engineer the features, then prepare them
def make_pipeline(estimator, scale=False, step_name='model'):
    return Pipeline([
        ('engineer', FunctionTransformer(add_engineered, validate=False)),
        ('prep', make_preprocessor(scale=scale)),
        (step_name, estimator),
    ])

Now I split the members into two groups. The model learns from the first and is judged on the
second, which it never sees during training.

This is the only honest way to know whether a model has learnt something general or has simply
memorised the members it was shown. A model can always describe data it has already seen; the
question is whether it says anything useful about a member it has not.

In [ ]:
# Splitting into a group to learn from and a group to be judged on
X_train, X_test, y_train, y_test = train_test_split(
    df_ml, y_cost, test_size=TEST_SIZE, random_state=RANDOM_STATE)

y_high_train = y_high.loc[X_train.index]
y_high_test = y_high.loc[X_test.index]

pd.Series({'training members': len(X_train), 'held-out members': len(X_test)})

### A trap in the standard fix

Cost is lopsided, so a model trained on raw amounts chases the largest numbers — being wrong by
20,000 once outweighs being wrong by 300 many times. The usual fix is to fit on a log scale and
convert back.

**The conversion is the trap.** Fitting on logs and inverting returns the *median* of a group, not
the mean, and for skewed data the median is always lower. Every prediction comes back
systematically light.

Fitted both ways below. `mean_ratio` — total predicted over total actual — detects it. **It should
be 1.00.**

In [ ]:
# Shared settings for every boosted model below, so the comparisons are like for like
GBM_KW = dict(max_iter=500, learning_rate=0.06, max_leaf_nodes=31, min_samples_leaf=40,
              l2_regularization=1.0, early_stopping=True, n_iter_no_change=25,
              validation_fraction=0.1, random_state=RANDOM_STATE)


def log_target(pipeline):
    return TransformedTargetRegressor(regressor=pipeline, func=np.log1p,
                                      inverse_func=np.expm1, check_inverse=False)

In [ ]:
# Four models: a baseline that always guesses the average, two on the multiplied scale,
# and one that works directly in money using a loss built for lopsided amounts
candidates = {
    'baseline (always the mean)': make_pipeline(DummyRegressor(strategy='mean')),
    'ridge on log cost': log_target(make_pipeline(Ridge(alpha=1.0), scale=True)),
    'boosted trees on log cost': log_target(make_pipeline(HistGradientBoostingRegressor(**GBM_KW))),
    'boosted trees, gamma loss': make_pipeline(HistGradientBoostingRegressor(loss='gamma', **GBM_KW)),
}
list(candidates)

In [ ]:
# Fitting each one and measuring it on members it has never seen
median_cost = float(y_cost.median())
fitted, rows = {}, []

for name, estimator in candidates.items():
    fitted[name] = estimator.fit(X_train, y_train)
    pred = np.clip(fitted[name].predict(X_test), 1, None)
    rows.append({
        'model': name,
        'R2': r2_score(y_test, pred),
        'MAE': mean_absolute_error(y_test, pred),
        'MAE as % of median cost': mean_absolute_error(y_test, pred) / median_cost * 100,
        'mean_ratio': pred.mean() / y_test.mean(),
    })

results_A = pd.DataFrame(rows).set_index('model')
results_A.round(3)

##### What the four models say

**MAE** is the average miss in money, **R2** is variance accounted for, **mean_ratio** is the bias
check — and it decides this.

The **two log-scale models sit at 0.758 and 0.754**: the trap, exactly. They under-collect on the
whole book by a quarter, in one direction for everybody, so averaging across members cannot rescue
it.

**The gamma model sits at 0.989** with the highest R2 at 0.167, fitting directly in money with no
conversion step.

Its **worse MAE** (1,768 against 1,662) is a genuine trade: the log models are tuned to get the
middle member right, the gamma model the total. For pricing the total wins.

In [ ]:
# The standard correction for the conversion bias, worked out from the training residuals
log_model = fitted['boosted trees on log cost']
residuals = np.log1p(y_train) - log_model.regressor_.predict(X_train)
smearing = float(np.mean(np.exp(residuals)))

corrected = np.clip(log_model.predict(X_test), 1, None) * smearing

pd.Series({'correction factor': round(smearing, 3),
           'R2 after correction': round(r2_score(y_test, corrected), 4),
           'mean_ratio after correction': round(corrected.mean() / y_test.mean(), 3)})

Applying the standard correction to the biased model lifts its mean_ratio from 0.75 to about 0.99
and its R-squared to essentially the same place the gamma model reached on its own.

Two different repairs arriving at the same answer confirms the diagnosis was right. **I am keeping
the gamma model**, because it avoids the problem rather than correcting for it afterwards.

In [ ]:
# The chosen model, from here on
BEST_A = 'boosted trees, gamma loss'
cost_model = fitted[BEST_A]
pred_A = np.clip(cost_model.predict(X_test), 1, None)

round(float(r2_score(y_test, pred_A)), 4)

### What did the renewal-legal contract cost me?

Section 4.6 warned that not every utilisation field is knowable in advance, and the feature
contract acts on that. But a decision like that should be priced rather than assumed virtuous, so
here is the same model fitted with the same-period fields put back in.

In [ ]:
# The same gamma model, given the same-period fields the contract bars
same_period_cols = RENEWAL_COLS + list(SAME_PERIOD)
X_train_sp, X_test_sp = df.loc[X_train.index, same_period_cols], df.loc[X_test.index, same_period_cols]

sp_preprocessor = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'),
     [c for c in same_period_cols if c in df.select_dtypes(include=[np.number]).columns]),
    ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')),
                      ('onehot', OneHotEncoder(handle_unknown='infrequent_if_exist', sparse_output=False))]),
     [c for c in same_period_cols if c not in df.select_dtypes(include=[np.number]).columns]),
], remainder='drop')

sp_model = Pipeline([('prep', sp_preprocessor),
                     ('model', HistGradientBoostingRegressor(loss='gamma', **GBM_KW))]).fit(X_train_sp, y_train)
pred_sp = np.clip(sp_model.predict(X_test_sp), 1, None)

pd.DataFrame({
    'R2': [r2_score(y_test, pred_A), r2_score(y_test, pred_sp)],
    'MAE': [mean_absolute_error(y_test, pred_A), mean_absolute_error(y_test, pred_sp)],
    'mean_ratio': [pred_A.mean() / y_test.mean(), pred_sp.mean() / y_test.mean()],
}, index=['renewal-legal (the contract)', 'plus same-period fields']).round(3)

The same-period model scores better — it knows how often the member saw a doctor during the very
year being predicted — but by **0.002 of R2**, which is nothing.

Better than expected. I was ready to pay a real price for a contract that can run at renewal, and
there is barely anything to trade: the prior-history fields carry almost everything the
same-period fields do.

**The renewal-legal model carries forward everywhere below.**

### Is the model good enough to actually use?

An R-squared of 0.167 sounds poor, and for predicting an individual member it is — the typical
prediction misses by about 84% of what a typical member costs in a year. But that is not the only
question worth asking of a pricing model.

Insurance does not price individuals in isolation; it prices pools. So the fair test is whether the
model gets **groups** right, even though it cannot get individuals right. I sort the held-out
members into ten groups by what the model predicted, then compare each group's predicted average
against its actual average.

In [ ]:
# Sorting members into ten groups by predicted cost, then checking each group's average
decile = pd.qcut(pred_A, 10, labels=False, duplicates='drop')
by_decile = (pd.DataFrame({'predicted': pred_A, 'actual': y_test.values, 'decile': decile})
               .groupby('decile').agg(members=('actual', 'size'),
                                      predicted=('predicted', 'mean'),
                                      actual=('actual', 'mean')))
by_decile['error %'] = (by_decile.predicted / by_decile.actual - 1) * 100
by_decile.round(1)

In [ ]:
# The same check as a chart
plt.figure(figsize=(10, 6))
plt.plot(by_decile.index, by_decile.predicted, marker='o', lw=2.5, color=BLUE, label='Predicted average')
plt.plot(by_decile.index, by_decile.actual, marker='o', lw=2.5, color=ORANGE, label='Actual average')
plt.xlabel('Group, ordered by what the model predicted')
plt.ylabel('Average annual medical cost')
plt.title('The model gets groups right, even though it cannot get individuals right')
plt.legend()
plt.grid(alpha=0.5)
save('v3_01_cost_model_by_group')
plt.show()

##### Reading the decile check

Every group's predicted average lands within a few percent of actual, with errors in both
directions rather than one. The ordering is right — the group predicted cheapest is cheapest, the
group predicted dearest is dearest.

Better than the R2 suggested, and this is the number for an actuary. Model A cannot say what an
individual will cost; it can sort members into groups and price each group. That is what a
technical premium needs. **ACT**

### Validation that is not a single split

One random split is one draw. Before I would let anybody use this I want to know how much the score
moves when the split moves, whether the model is memorising the training members, and whether more
data would still help.

In [ ]:
# Five-fold cross-validation, plus the gap between training and held-out performance
cv_r2 = cross_val_score(make_pipeline(HistGradientBoostingRegressor(loss='gamma', **GBM_KW)),
                        df_ml, y_cost, cv=KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
                        scoring='r2', n_jobs=N_JOBS)
train_pred = np.clip(cost_model.predict(X_train), 1, None)

pd.Series({
    'CV R2, mean': round(float(cv_r2.mean()), 3),
    'CV R2, standard deviation': round(float(cv_r2.std()), 3),
    'Training R2': round(r2_score(y_train, train_pred), 3),
    'Held-out R2': round(r2_score(y_test, pred_A), 3),
    'Overfit gap (train minus test)': round(r2_score(y_train, train_pred) - r2_score(y_test, pred_A), 3),
})

In [ ]:
# Does more data still help, or is the model already at its ceiling?
sizes, train_scores, val_scores = learning_curve(
    make_pipeline(HistGradientBoostingRegressor(loss='gamma', **GBM_KW)),
    df_ml, y_cost, cv=3, scoring='r2', train_sizes=np.linspace(0.25, 1.0, 4), n_jobs=1)

pd.DataFrame({'training members': sizes.astype(int),
              'training R2': train_scores.mean(axis=1).round(3),
              'cross-validated R2': val_scores.mean(axis=1).round(3)})

The cross-validated score sits close to the single-split figure with a small standard deviation, so
the result is not an artefact of one lucky split. The training-to-test gap is modest, so the model
is not memorising.

The learning curve is the more interesting one. The cross-validated score flattens well before the
full dataset, which says **more members of the same kind would not help.** The ceiling is
information, not sample size, and section 4.2 already told me where that ceiling is. If a future
version scores much higher, the first thing to check is whether a barred field has crept back in.

### The bad year, modelled directly

Section 4.8 argued that the thing insurance actually absorbs is the bad year, not the average one.
A model predicting the mean answers the wrong question for that story, so I fit two quantile models
alongside: one for the typical year and one for the bad year.

In [ ]:
# Quantile models: the middle member of a group, and the member nine tenths of the way along
quantile_models = {}
for q in (0.5, 0.9):
    quantile_models[q] = make_pipeline(
        HistGradientBoostingRegressor(loss='quantile', quantile=q, **GBM_KW)).fit(X_train, y_train)

p50_pred = quantile_models[0.5].predict(X_test)
p90_pred = quantile_models[0.9].predict(X_test)

pd.Series({
    'p50 coverage (should be near 50%)': round((y_test <= p50_pred).mean() * 100, 1),
    'p90 coverage (should be near 90%)': round((y_test <= p90_pred).mean() * 100, 1),
    'median p50 prediction': round(float(np.median(p50_pred))),
    'median p90 prediction': round(float(np.median(p90_pred))),
})

Coverage is the check that matters. A p90 model is calibrated when roughly 90% of members really do
come in below its prediction, and that is what happens here.

Practically, this gives product a defensible answer to "what should this member expect in a bad
year", which is the question 4.8 said they should actually be selling on. **PROD / ACT**

### Constraints an actuary can defend, and what the model leans on

A boosted tree will happily learn that cost *falls* between two adjacent ages if the sample happens
to say so. Nobody can defend that in a pricing committee, and it is not a real effect. So I
constrain four fields to be non-decreasing: age, chronic burden and the two prior-inpatient
measures. No constraint is placed on anything else.

In [ ]:
# Monotonic constraints: cost must not fall as age, chronic burden or prior admissions rise
prepared = make_preprocessor().fit(add_engineered(X_train))
feature_names = list(prepared.get_feature_names_out())

MONOTONIC_UP = {'Age', 'Chronic Conditions Count', 'Hospitalizations in Last 3 Years',
                'Days Hospitalized in Last 3 Years'}
constraints = [1 if name in MONOTONIC_UP else 0 for name in feature_names]

mono_model = make_pipeline(
    HistGradientBoostingRegressor(loss='gamma', monotonic_cst=constraints, **GBM_KW)).fit(X_train, y_train)
pred_mono = np.clip(mono_model.predict(X_test), 1, None)

pd.Series({'R2 unconstrained': round(r2_score(y_test, pred_A), 4),
           'R2 with constraints': round(r2_score(y_test, pred_mono), 4)})

The constraints cost almost nothing, which is the answer I hoped for: the shape they enforce is the
shape the data already had, and imposing it buys defensibility for free.

Now, what is the model actually using? SHAP attributes each prediction back to the features that
produced it, so the answer is measured rather than inferred.

In [ ]:
# SHAP on a sample, since the full test set is unnecessary for a summary
shap_sample = X_test.sample(2000, random_state=RANDOM_STATE)
shap_values = shap.TreeExplainer(mono_model.named_steps['model']).shap_values(
    prepared.transform(add_engineered(shap_sample)))

plt.figure(figsize=(9, 7))
shap.summary_plot(shap_values, features=None, feature_names=feature_names,
                  plot_type='bar', max_display=15, show=False)
plt.title('What the cost model runs on')
plt.tight_layout()
save('v3_04_shap_summary')
plt.show()

Chronic burden and the engineered comorbidity load dominate, followed by prior inpatient history
and age. That is the same ranking section 4.2 found by a completely different route, which is
reassuring — the model has not discovered anything the exploratory work missed, and it has not
latched onto something spurious either.

Demographic fields sit near the bottom. That is a fairness observation as much as a technical one,
and I test it properly in 6.3.

### The actuarial decomposition: frequency times severity

One more view of the same target, because it is the one an insurance audience reads most naturally.
Pure premium is claim frequency multiplied by claim severity, and fitting them separately produces
**rate relativities** — a multiplier per risk factor — rather than a black box.

`Claims Count` is the target of the frequency model here, not a feature of it.

In [ ]:
# Frequency: a Poisson model on claim count. Severity: a gamma model on cost per claim.
FS_FEATURES = ['Age', 'Chronic Conditions Count', 'Hospitalizations in Last 3 Years',
               'Days Hospitalized in Last 3 Years', 'BMI', 'Systolic Blood Pressure', 'HbA1c Level']


def design_matrix(index):
    frame = df.loc[index, FS_FEATURES + ['Smoker Status']]
    encoded = pd.get_dummies(frame, columns=['Smoker Status'], drop_first=True).astype(float)
    return sm.add_constant(encoded)


X_glm_train = design_matrix(X_train.index)
claims_train = df.loc[X_train.index, 'Claims Count']

freq_glm = sm.GLM(claims_train, X_glm_train, family=sm.families.Poisson()).fit()

claimed = claims_train > 0
sev_glm = sm.GLM(y_train[claimed] / claims_train[claimed], X_glm_train[claimed],
                 family=sm.families.Gamma(sm.families.links.Log())).fit()

pd.DataFrame({
    'frequency relativity': np.exp(freq_glm.params).drop('const'),
    'severity relativity': np.exp(sev_glm.params).drop('const'),
}).assign(**{'pure premium relativity': lambda d: d.iloc[:, 0] * d.iloc[:, 1]}).round(3)

Each number is a multiplier on the base rate for a one-unit increase in that factor. A frequency
relativity of 1.02 on chronic conditions means each additional condition raises expected claim
count by 2%.

This is the format a pricing committee can actually argue with, which the boosted model is not.
Both are fitted on the same renewal-legal contract, so they are describing the same book.

In [ ]:
# Does the decomposition agree with the boosted model at group level?
X_glm_test = design_matrix(X_test.index)
pure_premium = freq_glm.predict(X_glm_test) * sev_glm.predict(X_glm_test)

fs_check = pd.DataFrame({'actual': y_test.values, 'predicted': pure_premium.values})
fs_check['decile'] = pd.qcut(fs_check['predicted'], 10, labels=False)
fs_group = fs_check.groupby('decile').agg(actual_mean=('actual', 'mean'),
                                          predicted_mean=('predicted', 'mean')).round(0)
fs_group['error %'] = ((fs_group['predicted_mean'] / fs_group['actual_mean'] - 1) * 100).round(1)
fs_group

The decomposition orders groups in the same direction as the boosted model, which is the agreement
worth having. But its group errors are **much** larger: 20-28% under on the cheap deciles, **52%
over** on the most expensive, against a few percent for the gamma model.

That is the price of a form simple enough to hand over as a rate table, and it is too high to price
on. Keep the relativities for the argument they support — which factors move frequency, which move
severity — and use Model A for numbers that must be right. **ACT**

---

## 6.2 Model B — Who Becomes Expensive?

This is the model with a real job to do. Care management can only enrol a limited number of
members, so they need those members ranked by risk **before** the costs happen.

I define "high cost" as landing above the 90th percentile — the most expensive tenth of the book.
That is a deliberate choice rather than an obvious one: it matches the group section 4.6 found
carries a third of all spend, and a tenth of the membership is a plausible size for a programme.

In [ ]:
# What counts as a high-cost member
pd.Series({'threshold cost': round(float(y_cost.quantile(TAIL_Q))),
           'share of the held-out group above it': round(float(y_high_test.mean()), 3)})

Anyone costing more than about 6,270 counts as high cost, and they are 10% of the held-out group —
which is what defining it by the 90th percentile guarantees.

That 10% matters for how I judge the model. **A model that simply predicted "not high cost" for
everybody would be right 90% of the time**, so accuracy is a useless measure here. I need measures
that account for how rare the thing being predicted is.

In [ ]:
# The classifier. class_weight="balanced" tells it to treat the rare high-cost members
# as being as important as the common ordinary ones.
risk_model = make_pipeline(
    HistGradientBoostingClassifier(class_weight='balanced', **GBM_KW)).fit(X_train, y_high_train)

proba = risk_model.predict_proba(X_test)[:, 1]

pd.Series({
    'roc_auc': roc_auc_score(y_high_test, proba),
    'pr_auc': average_precision_score(y_high_test, proba),
    'brier': brier_score_loss(y_high_test, proba),
    'prevalence': float(y_high_test.mean()),
}).round(4)

##### Interpretation

**ROC-AUC 0.757.** Given one high-cost and one ordinary member, the chance the model scores the
high-cost one higher. It gets that right about three times in four.

**PR-AUC 0.265** against a 0.10 base rate — the honest measure when the target is rare, and roughly
2.6 times random. This is the figure to quote.

**Brier 0.195** measures whether the probabilities themselves are trustworthy, lower being better.
It needs a comparison to mean anything.

In [ ]:
# What a model that just predicts the base rate for everyone would score
baseline_brier = brier_score_loss(y_high_test, np.full(len(y_high_test), y_high_train.mean()))
round(baseline_brier, 4)

**The baseline scores 0.091, beating the model's 0.195.** Alarming, and it does not mean the model
is useless.

`class_weight='balanced'` is the cause. Treating rare high-cost members as equally important stops
the model ignoring them, at the cost of **inflated probabilities** — it says 70% where the true
chance is nearer 30%.

Inflation does not harm the **ranking**, which is what a targeting list needs. It does mean the
numbers cannot be read as probabilities, and anyone told "70% chance" would be misled.

Calibration fixes it, fitted on training members with internal cross-validation — fitting on the
held-out set and scoring it would flatter the result.

In [ ]:
# Calibrating on the training members only, with internal cross-validation
calibrated_model = CalibratedClassifierCV(clone(risk_model), method='isotonic', cv=3, n_jobs=N_JOBS)
calibrated_model.fit(X_train, y_high_train)
proba_cal = calibrated_model.predict_proba(X_test)[:, 1]

pd.Series({
    'Brier, raw model': brier_score_loss(y_high_test, proba),
    'Brier, calibrated': brier_score_loss(y_high_test, proba_cal),
    'Brier, always predicting the base rate': baseline_brier,
}).round(4)

In [ ]:
# How well the probabilities match reality, before and after calibration
plt.figure(figsize=(10, 6))
for label, p, colour in [('Raw model', proba, BLUE), ('After calibration', proba_cal, ORANGE)]:
    observed, predicted = calibration_curve(y_high_test, p, n_bins=10, strategy='quantile')
    plt.plot(predicted, observed, marker='o', lw=2.5, color=colour, label=label)

plt.plot([0, 1], [0, 1], ls='--', color='grey', label='Perfect (prediction = reality)')
plt.xlabel('Probability the model predicted')
plt.ylabel('Share who actually turned out to be high cost')
plt.title('The raw model overstates risk; calibration corrects it')
plt.legend()
plt.grid(alpha=0.5)
save('v3_02_risk_model_calibration')
plt.show()

##### Reading the calibration curve

The dashed line is honesty: say 30%, and 30% of those members are high cost.

The raw model runs well **below** it — where it says 60%, about 30% turn out expensive. Systematic
overstatement, as the Brier score indicated.

Calibrated, the line tracks the diagonal and Brier falls to **0.083**, now beating the 0.091
baseline. Beating the baseline is the test that matters: the probabilities carry real information
about individuals rather than repeating the book-wide rate.

Two usable versions. For **ranking**, either works — calibration does not reorder. For **quoting a
probability** or sizing an enrolled group, the calibrated one is required.

### Turning the model into a decision

A model that outputs probabilities is not yet a decision. Somebody has to choose a cut-off, and the
usual default of 0.5 is the wrong way to make that choice — it is a statistical convention, not a
business one.

The real constraint is **capacity**. A care-management team can handle a certain number of members,
so the question is: if we enrol the top N by predicted risk, what do we get for it?

In [ ]:
# What enrolling the top N members by predicted risk actually buys
actual_cost_test = y_cost.loc[X_test.index]
capacity_rows = []

for pct in [0.02, 0.05, 0.10, 0.15, 0.20, 0.30]:
    n_enrol = int(len(y_high_test) * pct)
    idx = np.argsort(-proba_cal)[:n_enrol]
    caught = y_high_test.values[idx].sum()
    capacity_rows.append({
        'capacity %': pct * 100,
        'members enrolled': n_enrol,
        'precision': caught / max(n_enrol, 1),
        'recall': caught / max(int(y_high_test.sum()), 1),
        'lift': (caught / max(n_enrol, 1)) / y_high_test.mean(),
        '% of spend reached': actual_cost_test.values[idx].sum() / actual_cost_test.sum() * 100,
    })

pd.DataFrame(capacity_rows).round(3)

##### What the capacity table buys

**Precision** — enrolled members who really are high cost: 0.31 at 10% capacity, so 31 per hundred
enrolled. **Recall** — high-cost members caught: also about 0.31, so two thirds are missed.

The two always pull against each other, and choosing is a business call about the cost of missing
someone against enrolling someone unnecessarily.

**Lift** is roughly 3 — three times as many expensive members as picking at random.

**Share of spend reached** is the budget number: 10% of members puts about a fifth of total spend
inside the programme, against 33.5% for a perfect targeter.

Present the 2% row alongside: highest precision and lift, so a small intensive programme is most
efficient per member. **CM**

### Fixing the cut-off honestly

The threshold has to be derived on data the model was not judged on, and on the same probability
scale it will be applied to. Deriving it on the held-out set and then reporting performance at that
threshold, on the same set, would be marking my own homework — so I carve a validation split out of
the training members instead.

In [ ]:
# Deriving the operating threshold on a validation split of TRAIN, on the calibrated scale
X_inner, X_val, y_inner, y_val = train_test_split(
    X_train, y_high_train, test_size=0.2, random_state=RANDOM_STATE, stratify=y_high_train)

val_proba = calibrated_model.predict_proba(X_val)[:, 1]
n_enrol_val = int(len(X_val) * CAPACITY_PCT)
OPERATING_THRESHOLD = float(np.sort(val_proba)[::-1][n_enrol_val - 1])

enrolled = proba_cal >= OPERATING_THRESHOLD

pd.Series({
    'operating threshold (calibrated probability)': round(OPERATING_THRESHOLD, 4),
    'members enrolled in the held-out group': int(enrolled.sum()),
    'precision at capacity': round(precision_score(y_high_test, enrolled), 3),
    'recall at capacity': round(recall_score(y_high_test, enrolled), 3),
    'lift over the base rate': round(precision_score(y_high_test, enrolled) / y_high_test.mean(), 2),
})

In [ ]:
# The full breakdown at that cut-off. This is the one place a printed report has no display
# equivalent worth building, so it stays a print.
print(classification_report(y_high_test, enrolled, target_names=['ordinary', 'high cost'], digits=3))

In [ ]:
# Five-fold cross-validated AUC, so the ranking quality is not a single-split claim either
cv_auc = cross_val_score(make_pipeline(HistGradientBoostingClassifier(class_weight='balanced', **GBM_KW)),
                         df_ml, y_high, cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
                         scoring='roc_auc', n_jobs=N_JOBS)

pd.Series({'CV roc_auc mean': round(float(cv_auc.mean()), 3),
           'CV roc_auc standard deviation': round(float(cv_auc.std()), 3)})

### What is the model actually using?

Before trusting a model I want to know what it leans on, for two reasons. If it depends on
something that will not be available at prediction time, it fails in use. And if it depends on
something it should not be using — a proxy for sex or region, say — that is a fairness problem.

The test shuffles one column at random and measures how much worse the model gets. A column the
model relies on causes a large drop when scrambled; a column it ignores causes none.

In [ ]:
# Scrambling each column in turn to see how much the model depends on it
importance_sample = X_test.sample(min(5000, len(X_test)), random_state=RANDOM_STATE)
permutation = permutation_importance(
    risk_model, importance_sample, y_high_test.loc[importance_sample.index],
    n_repeats=5, random_state=RANDOM_STATE, scoring='roc_auc', n_jobs=N_JOBS)

importance = pd.Series(permutation.importances_mean, index=importance_sample.columns).sort_values(ascending=False)

top = importance.head(12)[::-1]
plt.figure(figsize=(10, 6))
plt.barh(range(len(top)), top.values, color=BLUE)
plt.yticks(range(len(top)), top.index)
plt.xlabel('Drop in performance when the column is scrambled')
plt.title('The risk model runs on four fields')
plt.grid(axis='x', alpha=0.5)
save('v3_03_risk_model_importance')
plt.show()

##### Interpretation

Four fields do nearly all the work: **chronic condition count** by a wide margin, then **smoking**,
**days in hospital** and **age**.

All four are known before the year starts, so the model can run at renewal — the feature contract
holding in practice, not just on paper.

Region, sex, marital status, education and income sit at the bottom near zero, consistent with 4.2
finding each explains under 1% of cost variance. The model is not using demographics as a back
door, largely because there is no signal to use.

An importance table is not a fairness audit, though, and it is not treated as one — 6.3 measures it.

---

## 6.3 Fairness, Measured Rather Than Asserted

A model can be blind to a demographic field and still perform unevenly across groups. So rather
than infer fairness from what the model leans on, I measure both models directly across sex, region
and age band: is the cost model biased within each group, and does the risk model rank equally well
inside each one?

In [ ]:
# Group-level performance of both models across the dimensions that matter
fairness = df.loc[X_test.index, ['Sex', 'Region']].copy()
fairness['age band'] = pd.cut(df.loc[X_test.index, 'Age'], [0, 29, 49, 64, 200],
                              labels=['<=29', '30-49', '50-64', '65+'])
fairness['actual_cost'] = y_test.values
fairness['predicted_cost'] = pred_A
fairness['actual_high'] = y_high_test.values
fairness['proba_high'] = proba_cal

fairness_rows = []
for dimension in ['Sex', 'Region', 'age band']:
    for group, sub in fairness.groupby(dimension, observed=True):
        fairness_rows.append({
            'dimension': dimension, 'group': group, 'members': len(sub),
            'cost model bias %': round((sub['predicted_cost'].mean() / sub['actual_cost'].mean() - 1) * 100, 1),
            'risk model AUC': round(roc_auc_score(sub['actual_high'], sub['proba_high']), 3)
            if sub['actual_high'].nunique() > 1 else np.nan,
        })

pd.DataFrame(fairness_rows)

Every group's cost bias sits within a few points of zero, and the risk model's ranking quality is
stable across sex and region. The age bands vary a little more, which is expected: the model relies
on age, and the oldest band is the smallest.

This is measured now rather than inferred from an importance table. Before any deployment it should
be rerun on the actual enrolment list rather than on a random test split, and the affordability
impact from section 4.4 reviewed with compliance. **UW / PROD**

## 6.4 Who Pays More Than Their Risk?

Section 4.1 ruled out any pricing-adequacy claim, and that stands. But a company would still
reasonably ask which members contribute more than their risk-based cost and which contribute less,
and Model A gives the only honest way to look at that on this data: compare what a member actually
pays against what the model expects them to cost.

This is descriptive. It is not a pricing recommendation, because the premium field's provenance
forbids one.

In [ ]:
# Actual contribution against model-expected cost, by age and chronic burden
subsidy = df.loc[X_test.index, ['Age', 'Annual Premium', 'Chronic Conditions Count']].copy()
subsidy['expected_cost'] = pred_A
subsidy['age band'] = pd.cut(subsidy['Age'], [15, 29, 49, 64, 200], labels=['16-29', '30-49', '50-64', '65+'])
subsidy['chronic band'] = pd.cut(subsidy['Chronic Conditions Count'], [-1, 0, 1, 99],
                                 labels=['0 conditions', '1 condition', '2+ conditions'])

cross = subsidy.groupby(['age band', 'chronic band'], observed=True).agg(
    members=('Annual Premium', 'count'),
    actual_contribution=('Annual Premium', 'mean'),
    expected_cost=('expected_cost', 'mean')).round(0)
cross['ratio actual / expected'] = (cross['actual_contribution'] / cross['expected_cost']).round(2)
cross

In [ ]:
# The same thing as a map, which is easier to read than twelve rows
pivot = cross.reset_index().pivot(index='chronic band', columns='age band', values='ratio actual / expected')

fig, ax = plt.subplots(figsize=(9, 4))
im = ax.imshow(pivot.values.astype(float), cmap=DIV, vmin=0, vmax=float(np.nanmax(pivot.values)) * 2)
ax.set_xticks(range(len(pivot.columns)), pivot.columns)
ax.set_yticks(range(len(pivot.index)), pivot.index)
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        ax.text(j, i, f'{pivot.values[i, j]:.2f}x', ha='center', va='center', fontsize=11)
ax.set_title('Actual contribution divided by model-expected cost')
plt.tight_layout()
save('v3_05_cross_subsidy')
plt.show()

The ratios are strikingly uniform, and that uniformity **is** the finding. Because the actual
formula charges a fixed share of realised cost, and the model prices expected cost from risk
factors, every segment lands in the same narrow band. No group is meaningfully subsidising another
— which is exactly what you would expect from a contribution formula that reads the answer rather
than assessing risk. **ACT / FIN**

---

## 6.5 Why I Am Not Modelling the Premium

I said at the start I was refusing to build a premium model. Here is the evidence, so the omission
is visibly a choice rather than an oversight.

In [ ]:
# Reproducing the premium from the formula recovered in 4.1
formula_premium = 200 + 0.01 * df['Deductible'] + df['Network Tier'].map(RATE) * df[TARGET]
formula_error = (df['Annual Premium'] - formula_premium).abs()

pd.Series({
    'R2 of the formula': r2_score(df['Annual Premium'], formula_premium),
    'Largest error anywhere': formula_error.max(),
    'Share reproduced to the cent': (formula_error <= 0.005).mean(),
}).round(6)

The formula reproduces the premium with an R-squared of 1.000000 and a largest error anywhere of
half a cent, which is what rounding to the nearest cent produces.

A model trained on this target would score near-perfectly and would have learnt nothing about risk.
The premium here is a **contribution calculated after the cost is known**, not a price set in
advance, so predicting it is predicting arithmetic.

The genuinely useful question — what should we charge a member — is answered by predicting what
they will cost, which is Model A.

---

## 6.6 Saving the Models, and Proving They Work

Three models leave this notebook: the cost model, the bad-year model and the calibrated risk model.
Each gets a description written alongside it, so a `.pkl` is never an unlabelled binary.

Then I test that they actually load and score raw member records, because a model that cannot be
reloaded is not a deliverable.

In [ ]:
# Writing a description alongside each saved model
def save_model(name, estimator, task, target, metrics, intended_use, not_for, extra=None):
    joblib.dump(estimator, MODELS / f'{name}.pkl', compress=3)
    meta = {
        'model_name': name,
        'created_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
        'sklearn_version': sklearn.__version__,
        'pandas_version': pd.__version__,
        'random_state': RANDOM_STATE,
        'task': task,
        'target': target,
        'n_train': len(X_train),
        'n_test': len(X_test),
        'features': list(df_ml.columns),
        'engineered_inside_pipeline': ENGINEERED_NAMES,
        'excluded_leaky': list(LEAKY),
        'excluded_same_period': list(SAME_PERIOD),
        'metrics': metrics,
        'intended_use': intended_use,
        'not_for': not_for,
        **(extra or {}),
    }
    (MODELS / f'{name}_metadata.json').write_text(json.dumps(meta, indent=2, default=str))
    return name

In [ ]:
# Saving all three
save_model('costA_renewal_gamma', mono_model,
           'regression (gamma loss, monotonic, renewal-legal features)', TARGET,
           {'r2': float(r2_score(y_test, pred_mono)),
            'mae': float(mean_absolute_error(y_test, pred_mono)),
            'mean_ratio': float(pred_mono.mean() / y_test.mean()),
            'cv_r2_mean': float(cv_r2.mean()), 'cv_r2_sd': float(cv_r2.std())},
           'Expected cost for groups and segments; the basis for a technical premium. Runnable at renewal.',
           'Pricing or underwriting decisions about an individual member.')

save_model('costQ_p90', quantile_models[0.9],
           'quantile regression (90th percentile bad-year cost)', TARGET,
           {'p90_coverage': float((y_test <= p90_pred).mean())},
           "Estimating a member's bad-year exposure for product and group planning.",
           'Individual member pricing.')

save_model('riskB_renewal_calibrated', calibrated_model,
           'classification (calibrated probabilities)', f'{TARGET} above the {int(TAIL_Q * 100)}th percentile',
           {'roc_auc': float(roc_auc_score(y_high_test, proba_cal)),
            'brier': float(brier_score_loss(y_high_test, proba_cal)),
            'brier_baseline': float(baseline_brier),
            'cv_auc_mean': float(cv_auc.mean()),
            'precision_at_capacity': float(precision_score(y_high_test, enrolled)),
            'recall_at_capacity': float(recall_score(y_high_test, enrolled))},
           'Ranking members for care-management enrolment at a fixed capacity.',
           'Pricing, refusing cover, loading a premium, or any decision that disadvantages a member.',
           extra={'operating_threshold': OPERATING_THRESHOLD,
                  'threshold_scale': 'calibrated probability',
                  'threshold_derived_on': '20% stratified validation split of the training members',
                  'capacity_pct': CAPACITY_PCT})

sorted(p.name for p in MODELS.iterdir())

In [ ]:
# Reloading from disk and scoring untouched rows straight from the cleaned CSV
def load_and_predict(name, raw_rows):
    model = joblib.load(MODELS / f'{name}.pkl')
    meta = json.loads((MODELS / f'{name}_metadata.json').read_text())
    features = raw_rows[meta['features']]
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(features)[:, 1], meta
    return model.predict(features), meta


raw_rows = pd.read_csv(CLEAN_PATH).head(5)
cost_pred, _ = load_and_predict('costA_renewal_gamma', raw_rows)
bad_year_pred, _ = load_and_predict('costQ_p90', raw_rows)
risk_pred, risk_meta = load_and_predict('riskB_renewal_calibrated', raw_rows)

pd.DataFrame({
    'Id': raw_rows['Id'],
    'actual cost': raw_rows[TARGET].round(0),
    'predicted cost': np.round(cost_pred),
    'predicted bad year': np.round(bad_year_pred),
    'risk of top decile': np.round(risk_pred, 3),
    'flag for care management': risk_pred >= risk_meta['operating_threshold'],
})

All three load from disk and score raw member records with no preparation, confirming the decision
to bundle engineering and encoding inside the pipeline. A recipient needs the cleaned CSV schema
and nothing else.

The predictions also make the individual-level limit concrete: member 0 cost 6,938 and the model
predicted 2,946. Not a bug — that is R2 0.167 inspected one member at a time, and why the model
card restricts Model A to groups.

---

## 6.7 Model Card

### Model A — expected cost (`costA_renewal_gamma.pkl`)

| | |
|:---|:---|
| **Does** | Predicts a member's annual cost |
| **Use for** | Group and segment expected cost; the basis for a technical premium |
| **Not for** | Pricing or underwriting an individual member |
| **Why not** | Typical prediction misses by about 84% of a typical annual cost |
| **Good at** | Every predicted-cost group within a few percent of actual, no directional bias |
| **Inputs** | Renewal-legal attributes plus eleven engineered features, computed in-pipeline |
| **Barred** | Premium, claims paid, average claim, loss ratio, risk score (computed from the answer); claims count, visits, procedure counts, major-procedure flag (same-period) |
| **Constraints** | Non-decreasing in age, chronic burden, prior inpatient history |

### Model B — high-cost risk (`riskB_renewal_calibrated.pkl`)

| | |
|:---|:---|
| **Does** | Ranks members by chance of landing in the most expensive tenth |
| **Use for** | Care-management enrolment, cut-off set by capacity |
| **Not for** | Pricing, refusing cover, loading a premium, or disadvantaging a member |
| **How good** | ~3× random. 10% enrolled reaches about a fifth of spend |
| **Probabilities** | Calibrated; threshold derived on a validation split, on the calibrated scale |
| **Fairness** | Cost bias within a few points, stable ranking across sex, region, age band — measured in 6.3. Re-audit on the real enrolment list before going live |

### Model C — bad-year cost (`costQ_p90.pkl`)

The 90th percentile rather than the mean — the number 4.8 argued product should sell on. Coverage
checks out near 90%. Group planning only.

### Limits on all three

**No time dimension.** One snapshot, one year per member, so these relate attributes to
same-period cost rather than forecasting next year. Deployment needs several years of history and
a test on a later year. The renewal-legal contract is what makes that an upgrade rather than a
rewrite.

**The ceiling is genuine.** R² 0.167 on money, 0.165 under cross-validation with a standard
deviation of 0.004. Section 4.2 put the ceiling near 16% independently. A much higher score means
checking the feature contract first.

**No model here should touch the alcohol field** — its largest category is entirely imputed.

---

## How to Run This

```bash
jupyter nbconvert --to notebook --execute --inplace notebook.ipynb
```

Reads `data/medical_insurance.csv`. Writes three things: the cleaned CSV, the figures under
`reports/figures/`, and three models with metadata under `updated_models/`.

That folder is deliberately separate from `models/`, which holds artefacts from the previous
notebook — some built on the same-period feature set and so unusable at renewal. Nothing here reads
or overwrites them.

Everything else — every table, the cleaning log, the bias report, the dashboard charts — stays in
the notebook. Fitting takes a few minutes; cross-validation, the learning curve and permutation
importance are the slow parts.